In [1]:
import os
import sys
import subprocess
import glob

# 🎛️ SET THIS TO TRUE FOR TPU, FALSE FOR GPU
FORCE_TPU = True

def repair_environment():

    if FORCE_TPU:
        print("🔍 Starting High-Speed TPU Repair...")

        # 1. Faster Uninstallation
        print("🧹 Wiping libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torch_xla", "torchvision", "numpy", "tensorflow", "huggingface_hub"],
                       capture_output=True)

        # 2. Parallel/Bulk Installation
        print("📥 Installing Synced TPU Stack...")
        common_args = ["install", "-q", "--no-warn-script-location"]

        if FORCE_TPU or glob.glob("/dev/accel*"):
            cmd = [
                sys.executable, "-m", "pip", *common_args,
                "torch==2.8.0",
                "torchvision==0.23.0",
                "torch_xla[tpu]==2.8.0",
                "numpy", "pyarrow==16.1.0", "fsspec", # <-- Pinned pyarrow here
                "protobuf>=5.28.0",
                "datasets>=2.20.0", "transformers", "huggingface_hub>=0.28.0", "wandb", # <-- Added >=2.20.0 to datasets
                "cloud-tpu-client", "scikit-learn", "pandas<3.0.0",
                "-f", "https://storage.googleapis.com/libtpu-releases/index.html",
                "--extra-index-url", "https://download.pytorch.org/whl/cpu"
            ]
            subprocess.check_call(cmd)
        else:
            # Fallback
            cmd = [
                sys.executable, "-m", "pip", *common_args, "-U",
                "torch", "datasets", "pyarrow", "transformers", "huggingface_hub>=0.28.0", "fsspec", "wandb", "scipy", "numpy", "pandas<3.0.0"
            ]
            subprocess.check_call(cmd)

        print("\n✅ TPU REPAIR COMPLETE.")
        print("⚠️ Click 'Run' -> 'Restart Session' NOW.")

    else:
        print("🔍 Starting Robust GPU Repair...")

        # 1. Clean Wipe
        print("🧹 Wiping conflicting libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torchvision", "torchaudio", "huggingface_hub"],
                       capture_output=True)

        # 2. Setup Arguments
        common_args = ["install", "-q", "--no-warn-script-location"]

        try:
            print("📥 Installing GPU/CUDA Stack...")

            print("   ⚡ Part 1: PyTorch Core...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args,
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu121"
            ])

            print("   ⚡ Part 2: Transformers & Data...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args, "-U",
                "datasets", "transformers", "huggingface_hub>=0.28.0",
                "wandb", "pandas<3.0.0"
            ])

            print("\n✅ GPU REPAIR COMPLETE.")
            print("⚠️ MANDATORY: Click 'Run' -> 'Restart Session' NOW.")

        except subprocess.CalledProcessError as e:
            print(f"\n❌ Installation failed. Error: {e}")
            print("💡 Try manually restarting the session and running this cell again.")

if __name__ == "__main__":
    repair_environment()

🔍 Starting High-Speed TPU Repair...
🧹 Wiping libraries...
📥 Installing Synced TPU Stack...



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



✅ TPU REPAIR COMPLETE.
⚠️ Click 'Run' -> 'Restart Session' NOW.


In [2]:
%%writefile model.py

##################################################
# Defines HELM Phase 13A: Elastic Threshold Router
# Has a total of 32 heads, d_head = 64; only 16 will be used at a time
# True Decoupling of d_model = d_head * n_head
# Acheived via expansion layer
# Target 8 16 32
# 4 Perm heads
# No Dead Head Penalty
# Still Use Clamp
##################################################

import os
import json
import torch
import numpy as np
from safetensors.torch import load_file
import math
from math import sqrt
import random
import torch.nn.functional as F
import torch.nn as nn
try:
    from torch_xla.utils.checkpoint import checkpoint as _xla_checkpoint
except Exception:
    _xla_checkpoint = None
from transformers import AutoTokenizer
from transformers import PretrainedConfig, PreTrainedModel



# modified justnorm() function
# better than F.normalize(), max() causes micro walls during gradient descent
# better than nGPT's version, prevents division by 0 error
def justnorm(x, dim = -1, eps = 1e-12):
    res = x / (x.norm(p=2, dim=dim, keepdim=True) + eps)
    return res

# Cast the input to the correct input layer dtype
def cast_linear(x, layer):
    w = layer.weight.to(x.dtype)
    b = None if layer.bias is None else layer.bias.to(x.dtype)
    return F.linear(x,w,b)


# Hugging Face Config Class (for future deployment)
class HELMConfig(PretrainedConfig):

    model_type = "helm_7d"

    def __init__(
        self,
        # General Model Hyperparameters
        hidden_size = 1024,
        sqrt_hidden_size = 32,
        max_position_embeddings = 4096,
        initializer_range = 0.03125,
        num_hidden_layers = 12,
        num_attention_heads = 32,
        d_head = 64,
        rope_theta = 160000,
        intermediate_size = 2816,
        norm_eps = 1e-12,
        hidden_act = "swiglu",
        swiglu_s_init = 1.0,
        base_lr = 3e-4,
        min_lr = 3e-5,
        weight_decay = 0.0,
        bias = False,
        use_ckpt = False,

        # Tokenization and Data Collator Hyperparameters
        tokenizer_path = "answerdotai/ModernBERT-base",
        vocab_size = 50368,
        bos_token_id = 50281,
        eos_token_id = 50282,
        pad_token_id = 50283,
        mask_token_id = 50284,
        unk_token_id = 50285,
        mlm_probability = 0.3,
        mlm_use_span_masking = True,
        mlm_span_length = 3,

        # HELM_7c Router
        num_router_latents = 4,
        num_permanent_heads = 8,
        head_target_min = 8,
        head_target_center = 16,
        head_target_max = 32,
        easiness_cdf_breakpoints = None,
        count_loss_lambda = 0.5,
        router_grad_clip = 0.05,

        # Permanent-head training noise
        jitter_noise = 0.01,

        # nGPT self attention and FFN hyperparameters
        ngpt_sqk_init_value = 1.0,
        ngpt_sqk_init_scale = 0.03125,
        use_exclusive_attention = True,
        ngpt_alpha_value_attn = 0.05,
        ngpt_alpha_scale_attn = 0.03125,
        ngpt_alpha_value_mlp = 0.05,
        ngpt_alpha_scale_mlp = 0.03125,
        ngpt_suv_value = 1.0,
        ngpt_suv_scale = 1.0,
        ngpt_sz_init_value = 1.00,
        ngpt_sz_init_scale = 0.03125,

        dataset_total_steps = 65000,
        **kwargs
    ):
        # General model
        self.hidden_size = hidden_size
        self.sqrt_hidden_size = sqrt_hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.d_head = d_head
        self.rope_theta = rope_theta
        self.intermediate_size = intermediate_size
        self.norm_eps = norm_eps
        self.hidden_act = hidden_act
        self.swiglu_s_init = swiglu_s_init
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.weight_decay = weight_decay
        self.bias = bias
        self.use_ckpt = use_ckpt

        # Tokenization / MLM
        self.tokenizer_path = tokenizer_path
        self.vocab_size = vocab_size
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.pad_token_id = pad_token_id
        self.mask_token_id = mask_token_id
        self.unk_token_id = unk_token_id
        self.mlm_probability = mlm_probability
        self.mlm_use_span_masking = mlm_use_span_masking
        self.mlm_span_length = mlm_span_length

        # HELM_7c Router
        self.num_router_latents = num_router_latents
        self.num_permanent_heads = num_permanent_heads
        self.head_target_min = head_target_min
        self.head_target_center = head_target_center
        self.head_target_max = head_target_max
        self.easiness_cdf_breakpoints = easiness_cdf_breakpoints
        self.count_loss_lambda = count_loss_lambda
        self.router_grad_clip = router_grad_clip
        self.jitter_noise = jitter_noise

        elastic = num_attention_heads - num_permanent_heads
        if num_permanent_heads != head_target_min:
            raise ValueError(
                "HELM_7c uses permanent heads as the structural minimum; "
                "num_permanent_heads must equal head_target_min."
            )
        if elastic <= 0:
            raise ValueError("HELM_7c requires at least one elastic head")
        if not (head_target_min <= head_target_center <= head_target_max <= num_attention_heads):
            raise ValueError("Invalid HELM_7c head targets")

        # nGPT
        self.ngpt_sqk_init_value = ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = ngpt_sqk_init_scale
        self.use_exclusive_attention = use_exclusive_attention
        self.ngpt_alpha_value_attn = ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = ngpt_alpha_scale_mlp
        self.ngpt_suv_value = ngpt_suv_value
        self.ngpt_suv_scale = ngpt_suv_scale
        self.ngpt_sz_init_value = ngpt_sz_init_value
        self.ngpt_sz_init_scale = ngpt_sz_init_scale
        self.dataset_total_steps = dataset_total_steps

        super().__init__(**kwargs)


class HELMEmbedding(nn.Module):

    # Initialize Embedding Layer
    def __init__(self, config):
        super().__init__()

        # Embedding Matrix size() : [vocab_size, hidden_size]
        self.word_embeddings = nn.Embedding(
            config.vocab_size,
            config.hidden_size,
            padding_idx=config.pad_token_id
        )

    # Forward Pass (yes, its literally 3 lines)
    def forward(self, input_ids):

        # Map input_ids from Word Embeddings
        word_embeds = self.word_embeddings(input_ids)

        # Normalize (an nGPT must to allow cos. sim. to work)
        embeddings = justnorm(word_embeds)

        # Return
        return embeddings



# HELM_7d multi-latent router
class HELMMultiViewRouter(nn.Module):
    """
    Minimal sequence-level elastic router for HELM_7d.

    Architecture:
      - 8 permanent heads
      - 24 elastic candidate heads

    Forward routing remains exactly the same as HELM_7c:
        hard_mask = 1[z_h > 0]

    The router itself is still trained with the original sigmoid STE:
        ste_mask = hard_mask.detach()
                 - sigmoid_scores.detach()
                 + sigmoid_scores

    HELM_7d adds a SECOND mask used only for the backward gradient
    received by the attention heads themselves:

        active elastic head   -> backward strength = 1.0
        inactive elastic head -> backward strength = sigmoid score

    This second mask is detached from the router graph so it does NOT
    introduce an additional router-gradient path. The router continues
    to receive the same CE/count-loss STE gradient it received in 7c.

    Easiness labels remain training-time supervision only. They are
    converted into a target total head count in [8, 32]. Inference does
    not require easiness.
    """

    def __init__(self, config):
        super().__init__()

        self.config = config
        self.scale = config.sqrt_hidden_size

        self.num_elastic_candidates = (
            config.num_attention_heads
            - config.num_permanent_heads
        )

        # ----------------------------------------------------------
        # Existing HELM multi-latent sequence summarizer
        # ----------------------------------------------------------
        self.q_down_proj = nn.Linear(
            config.hidden_size,
            config.num_router_latents,
            bias=config.bias,
        )

        self.l_i_weights = nn.Parameter(
            torch.ones(config.num_router_latents)
        )

        # ----------------------------------------------------------
        # Elastic head scorer
        #
        # As in HELM_7c, this is intentionally NOT normalized.
        # Router magnitude is allowed to carry information.
        # ----------------------------------------------------------
        self.q_up_proj = nn.Linear(
            config.hidden_size,
            self.num_elastic_candidates,
            bias=False,
        )

        # ----------------------------------------------------------
        # Last-forward telemetry
        # ----------------------------------------------------------
        self.save_router_logits = None
        self.save_sigmoid_scores = None
        self.save_hard_mask = None

        # HELM_7d-specific telemetry:
        # gradient strength that each elastic attention head itself
        # will receive during backward.
        self.save_head_backward_scores = None

        self.save_total_head_count = None
        self.save_target_total_head_count = None
        self.save_count_error = None
        self.save_count_loss = None

    def _easiness_to_target(self, easiness_score):
        """
        Map easiness label -> integer target total heads in [8, 32].

        Easiness is converted to a CDF quantile q so the target depends
        on relative difficulty rather than the raw numeric easiness value:

            q = 0.0   hardest -> 32 total heads
            q = 0.5   median  -> 16 total heads
            q = 1.0   easiest -> 8 total heads
        """

        batch = easiness_score.numel()
        device = easiness_score.device

        e = (
            easiness_score
            .to(torch.float32)
            .reshape(batch)
            .clamp(0.0, 1.0)
        )

        bp = self.config.easiness_cdf_breakpoints

        if bp is not None and len(bp) >= 2:
            breaks = torch.as_tensor(
                bp,
                device=device,
                dtype=torch.float32,
            )

            n_intervals = breaks.numel() - 1

            pos = torch.searchsorted(
                breaks,
                e,
                right=True,
            ).clamp(1, n_intervals)

            lo = breaks[pos - 1]
            hi = breaks[pos]

            frac = (
                (e - lo)
                / (hi - lo + 1e-8)
            )

            q = (
                (pos - 1).to(torch.float32)
                + frac
            ) / float(n_intervals)

            q = q.clamp(0.0, 1.0)

        else:
            # Safe fallback if no breakpoint table is supplied.
            q = e

        h_min = float(
            self.config.head_target_min
        )

        h_ctr = float(
            self.config.head_target_center
        )

        h_max = float(
            self.config.head_target_max
        )

        # ----------------------------------------------------------
        # Piecewise mapping:
        #
        # hardest half:
        #     32 -> 16
        #
        # easiest half:
        #     16 -> 8
        # ----------------------------------------------------------
        hard_half = q < 0.5

        hard_target = (
            h_ctr
            + (h_max - h_ctr)
            * ((0.5 - q) / 0.5)
        )

        easy_target = (
            h_ctr
            + (h_min - h_ctr)
            * ((q - 0.5) / 0.5)
        )

        target_total = torch.where(
            hard_half,
            hard_target,
            easy_target,
        )

        # Actual head counts are discrete, so supervise an exactly
        # attainable integer target.
        return (
            target_total
            .round()
            .clamp(h_min, h_max)
        )

    def forward(
        self,
        hidden_states,
        easiness_score=None,
    ):
        # ==========================================================
        # 1. Existing HELM multi-latent sequence summary
        # ==========================================================

        q_down = justnorm(
            self.q_down_proj.weight,
            dim=1,
        ).to(hidden_states.dtype)

        # [B, S, R]
        scanner = F.linear(
            hidden_states,
            q_down,
        )

        # Normalize scanner activations over sequence positions.
        # [B, S, R]
        scanner_weights = F.softmax(
            self.scale * scanner,
            dim=1,
        )

        # Weighted sequence summaries.
        # [B, R, D]
        latents = torch.bmm(
            scanner_weights.transpose(1, 2),
            hidden_states,
        )

        # Learn how strongly to combine the R router latents.
        latent_weights = F.softmax(
            self.l_i_weights,
            dim=0,
        )

        # Final sequence representation used by the head router.
        # [B, D]
        pooled = (
            latents
            * latent_weights.view(1, -1, 1)
        ).sum(dim=1)

        # ==========================================================
        # 2. Elastic head scores
        # ==========================================================

        # Raw learned routing scores.
        # [B, E]
        router_logits = cast_linear(
            pooled,
            self.q_up_proj,
        )

        # Continuous router confidence.
        # [B, E]
        sigmoid_scores = torch.sigmoid(
            router_logits
        )

        # Actual hard forward decision.
        # [B, E]
        hard_mask = (
            router_logits > 0
        ).to(router_logits.dtype)

        # ==========================================================
        # 3. HELM_7c router STE -- UNCHANGED
        # ==========================================================
        #
        # Forward:
        #     ste_mask == hard_mask
        #
        # Backward wrt router logits:
        #     d ste_mask / dz
        #       = sigmoid(z) * (1 - sigmoid(z))
        #
        # This continues to train the ROUTER exactly as in 7c.
        # ==========================================================

        ste_mask = (
            hard_mask.detach()
            - sigmoid_scores.detach()
            + sigmoid_scores
        )

        # ==========================================================
        # 4. HELM_7d attention-head backward strength
        # ==========================================================
        #
        # This is DIFFERENT from ste_mask.
        #
        # For the actual attention head parameters:
        #
        #     if head is ON:
        #         backward strength = 1.0
        #
        #     if head is OFF:
        #         backward strength = sigmoid confidence
        #
        # Examples:
        #
        #   p=.90, ON  -> 1.00
        #   p=.60, ON  -> 1.00
        #   p=.51, ON  -> 1.00
        #
        #   p=.49, OFF -> 0.49
        #   p=.30, OFF -> 0.30
        #   p=.05, OFF -> 0.05
        #
        # CRITICAL:
        # sigmoid_scores is DETACHED here.
        #
        # This mask is intended to change gradients into Q/K/V/O,
        # not create a second CE gradient path into the router.
        # ==========================================================

        hard_detached = hard_mask.detach()
        sigmoid_detached = sigmoid_scores.detach()

        head_backward_scores = (
            hard_detached
            + (1.0 - hard_detached)
            * sigmoid_detached
        )

        # ==========================================================
        # 5. Actual hard head count
        # ==========================================================

        actual_elastic_count = (
            hard_mask.sum(dim=-1)
        )

        actual_total_count = (
            actual_elastic_count
            + float(
                self.config.num_permanent_heads
            )
        )

        # ==========================================================
        # 6. Easiness-supervised hard-count loss
        # ==========================================================

        if easiness_score is not None:

            target_total_count = (
                self._easiness_to_target(
                    easiness_score
                )
            )

            target_elastic_count = (
                target_total_count
                - float(
                    self.config.num_permanent_heads
                )
            )

            # ------------------------------------------------------
            # The forward value of ste_mask is the ACTUAL hard count,
            # but its backward derivative follows sigmoid.
            #
            # This preserves the 7c protection against the old
            # sum(sigmoid) soft-count loophole.
            # ------------------------------------------------------

            differentiable_elastic_count = (
                ste_mask
                .float()
                .sum(dim=-1)
            )

            count_error = (
                differentiable_elastic_count
                - target_elastic_count.float()
            )

            denom = float(
                self.num_elastic_candidates
            )

            count_loss = (
                float(
                    self.config.count_loss_lambda
                )
                * (
                    count_error / denom
                ).square().mean()
            )

        else:
            if self.training:
                raise ValueError(
                    "HELM_7d training requires easiness_score"
                )

            target_total_count = (
                torch.full_like(
                    actual_total_count,
                    -1.0,
                )
            )

            count_error = (
                torch.zeros_like(
                    actual_total_count
                )
            )

            count_loss = (
                router_logits.new_zeros(())
            )

        # ==========================================================
        # 7. Telemetry
        # ==========================================================

        self.save_router_logits = (
            router_logits.detach()
        )

        self.save_sigmoid_scores = (
            sigmoid_scores.detach()
        )

        self.save_hard_mask = (
            hard_mask.detach()
        )

        self.save_head_backward_scores = (
            head_backward_scores.detach()
        )

        self.save_total_head_count = (
            actual_total_count.detach()
        )

        self.save_target_total_head_count = (
            target_total_count.detach()
        )

        self.save_count_error = (
            actual_total_count
            - target_total_count
        ).detach()

        self.count_loss = count_loss

        self.save_count_loss = (
            count_loss.detach()
        )

        # ==========================================================
        # 8. Construct the two full [B,H,1,1] masks
        # ==========================================================

        # ----------------------------------------------------------
        # router_mask
        #
        # Forward = hard 0/1 mask.
        # Backward = original HELM_7c STE.
        #
        # This is used ONLY for the router-gradient surrogate.
        # ----------------------------------------------------------
        router_mask = ste_mask.view(
            ste_mask.size(0),
            -1,
            1,
            1,
        )

        # ----------------------------------------------------------
        # head_backward_mask
        #
        # Active head   -> 1
        # Inactive head -> sigmoid probability
        #
        # Entire mask is detached from router graph.
        # ----------------------------------------------------------
        head_backward_mask = (
            head_backward_scores.view(
                head_backward_scores.size(0),
                -1,
                1,
                1,
            )
        )

        # Permanent heads remain fully active and fully trained.
        if self.config.num_permanent_heads > 0:

            permanent = torch.ones(
                ste_mask.size(0),
                self.config.num_permanent_heads,
                1,
                1,
                device=router_mask.device,
                dtype=router_mask.dtype,
            )

            router_mask = torch.cat(
                (
                    permanent,
                    router_mask,
                ),
                dim=1,
            )

            head_backward_mask = torch.cat(
                (
                    permanent,
                    head_backward_mask,
                ),
                dim=1,
            )

        # HELM_7d returns TWO masks instead of one.
        return router_mask, head_backward_mask


class RotaryEmbeddings(nn.Module):

    # Initialize the Following
    # rope_theta
    # max_position_embeddings
    # sin & cos table
    def __init__(self, dim, max_position_embeddings, rope_theta = 160000):
        super().__init__()

        # Define inverse of frequencies
        # size(): [dim/2]
        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2).float() / dim))

        # Create position vector
        # size(): [max_position_embeddings]
        t = torch.arange(max_position_embeddings, dtype = inv_freq.dtype)

        freqs = torch.outer(t, inv_freq)

        freqs = torch.cat((freqs, freqs), dim = -1)


        # Save the Sine and Cosine
        self.register_buffer("cos", freqs.cos())
        self.register_buffer("sin", freqs.sin())

    # Implement rotate_half (Allows for clean rotation mechanics)
    def rotate_half(self, x):

        # Take x as the first half
        x1 = x[..., : x.shape[-1] // 2]

        # Take y was the second half
        x2 = x[..., x.shape[-1] // 2 :]

        return torch.cat((-x2, x1), dim = -1)


    # Implement apply_rotary_embeddings
    # Does RoPE
    # Expected input size: [b, num_attention_heads, seq_len, dim]
    # Output: [b, num_attention_heads, seq_len, dim]
    def forward(self, x):

        # Get token length
        seq_len = x.shape[-2]

        # Take a slice of the cos and sin tables
        x_cos = self.cos[:seq_len, ...].to(dtype=x.dtype)
        x_sin = self.sin[:seq_len, ...].to(dtype=x.dtype)

        # Return RoPE matrix
        return (x * x_cos) + (self.rotate_half(x) * x_sin)



# Self Attention
# Literally Just Self Attention
# QKV cross self attention
# Use RoPE
# Output Matrix
# Specifics about training (masked training)
# MODIFICATION: USE FLEX ATTENTION TO ALLOW FOR BATCHED INFERENCE
#
# HELM_7d MODIFICATION:
# Hard-forward routing is unchanged from HELM_7c.
#
# However, during training:
#   - active heads receive full backward gradient
#   - inactive heads receive backward gradient proportional to sigmoid score
#
# This applies to BOTH:
#   - the attention head itself (Q/K/V)
#   - its corresponding columns in the output projection W_O
#
class HELMSelfAttention(nn.Module):

    # Initialize:
    #   - QKV matrix
    #   - Output matrix
    #   - Scaling vector sqk for q and k
    #   - RoPE Module
    def __init__(self, config):
        super().__init__()

        # ----------------------------------------------------------
        # Config convenience
        # ----------------------------------------------------------
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.num_permanent_heads = config.num_permanent_heads

        self.d_head = (
            config.d_head
            if config.d_head is not None
            else (
                config.hidden_size
                // config.num_attention_heads
            )
        )

        self.total_head_dim = (
            self.num_attention_heads
            * self.d_head
        )

        self.ngpt_sqk_init_value = (
            config.ngpt_sqk_init_value
        )

        self.ngpt_sqk_init_scale = (
            config.ngpt_sqk_init_scale
        )

        self.config = config

        # ----------------------------------------------------------
        # Eval backend
        # ----------------------------------------------------------
        self._eval_backend = "dense"
        self._flex_compiled = False
        self._flex_fn = None
        self._block_mask_fn = None

        # ----------------------------------------------------------
        # QKV Matrix
        #
        # hidden_size -> 3 * total_head_dim
        #
        # For HELM:
        #   1024 -> 3 * 2048
        # ----------------------------------------------------------
        self.qkv = nn.Linear(
            config.hidden_size,
            self.total_head_dim * 3,
            bias=config.bias,
        )

        # ----------------------------------------------------------
        # RoPE
        # ----------------------------------------------------------
        self.RoPE = RotaryEmbeddings(
            self.d_head,
            config.max_position_embeddings,
            config.rope_theta,
        )

        # ----------------------------------------------------------
        # SQK scalers
        # ----------------------------------------------------------
        self.sqk = nn.Parameter(
            self.ngpt_sqk_init_scale
            * torch.ones(
                self.total_head_dim
            )
        )

        # ----------------------------------------------------------
        # Output Matrix
        #
        # 2048 -> 1024
        # ----------------------------------------------------------
        self.output = nn.Linear(
            self.total_head_dim,
            config.hidden_size,
            bias=config.bias,
        )

    # ==============================================================
    # Eval backend setup
    # ==============================================================

    def set_eval_backend(
        self,
        backend="flex",
        compile=True,
    ):
        compile = bool(compile)

        changed = (
            backend
            != getattr(
                self,
                "_eval_backend",
                None,
            )
            or compile
            != getattr(
                self,
                "_flex_compiled",
                None,
            )
        )

        self._eval_backend = backend
        self._flex_compiled = compile

        if changed:
            self._flex_fn = None
            self._block_mask_fn = None

    def _flex_attn(
        self,
        q,
        k,
        v,
        block_mask,
        scale,
    ):
        if self._flex_fn is None:
            from torch.nn.attention.flex_attention import (
                flex_attention,
            )

            self._flex_fn = (
                torch.compile(
                    flex_attention
                )
                if self._flex_compiled
                else flex_attention
            )

        return self._flex_fn(
            q,
            k,
            v,
            block_mask=block_mask,
            scale=scale,
        )

    def _build_block_mask(
        self,
        mask_mod,
        B,
        H,
        S,
        device,
    ):
        if self._block_mask_fn is None:
            from torch.nn.attention.flex_attention import (
                create_block_mask,
            )

            self._block_mask_fn = (
                torch.compile(
                    create_block_mask
                )
                if self._flex_compiled
                else create_block_mask
            )

        return self._block_mask_fn(
            mask_mod,
            B,
            H,
            S,
            S,
            device=device,
        )

    # ==============================================================
    # HELM_7d training projection
    # ==============================================================

    def _helm7d_project(
        self,
        context_layer,
        router_mask,
        head_backward_mask,
        batch_size,
        seq_len,
    ):
        """
        HELM_7d hard-forward / soft-head-backward projection.

        context_layer:
            [B, H, S, d_head]

        router_mask:
            [B, H, 1, 1]

            Forward value:
                permanent -> 1
                elastic   -> exact hard 0/1

            Elastic router entries retain the original HELM_7c
            sigmoid STE gradient.

        head_backward_mask:
            [B, H, 1, 1]

            permanent -> 1

            elastic active:
                1

            elastic inactive:
                sigmoid(router_logit)

            This mask is detached from the router graph.

        Desired behavior
        ----------------

        FORWARD:

            y = W_O (m * C)

        exactly the same as HELM_7c.


        BACKWARD INTO ROUTER:

            same STE gradient as HELM_7c.


        BACKWARD INTO ATTENTION HEAD:

            active head   -> 1
            inactive head -> sigmoid score


        BACKWARD INTO W_O HEAD COLUMNS:

            active head   -> full training
            inactive head -> sigmoid-scaled training
        """

        # ----------------------------------------------------------
        # Expand masks
        # ----------------------------------------------------------

        hard_router = router_mask.expand_as(
            context_layer
        )

        # Safety: this path must NEVER create another gradient
        # into the router.
        backward_router = (
            head_backward_mask
            .detach()
            .expand_as(
                context_layer
            )
        )

        # ==========================================================
        # ROUTER SURROGATE PATH
        # ==========================================================
        #
        # We want the original HELM_7c CE gradient into the router:
        #
        #       context.detach() * STE_router_mask
        #
        # context is detached so this path trains ONLY the router.
        #
        # self.output weights are also detached so this path does
        # NOT train W_O.
        # ==========================================================

        router_context = (
            context_layer.detach()
            * hard_router
        )

        router_context = (
            router_context
            .permute(
                0,
                2,
                1,
                3,
            )
            .contiguous()
        )

        router_context = (
            router_context.view(
                batch_size,
                seq_len,
                -1,
            )
        )

        # Detach W_O for this path:
        # this path exists ONLY to preserve router gradients.
        router_weight = (
            self.output.weight
            .detach()
            .to(
                router_context.dtype
            )
        )

        router_bias = (
            None
            if self.output.bias is None
            else (
                self.output.bias
                .detach()
                .to(
                    router_context.dtype
                )
            )
        )

        router_projected = F.linear(
            router_context,
            router_weight,
            router_bias,
        )

        # ==========================================================
        # ATTENTION-HEAD SURROGATE PATH
        # ==========================================================
        #
        # This path trains:
        #
        #   Q
        #   K
        #   V
        #   W_O
        #
        # using:
        #
        #   active head:
        #       gradient multiplier = 1
        #
        #   inactive head:
        #       gradient multiplier = sigmoid score
        #
        # IMPORTANT:
        #
        # This path is numerically cancelled from the forward pass.
        # ==========================================================

        backward_context = (
            context_layer
            * backward_router
        )

        backward_context = (
            backward_context
            .permute(
                0,
                2,
                1,
                3,
            )
            .contiguous()
        )

        backward_context = (
            backward_context.view(
                batch_size,
                seq_len,
                -1,
            )
        )

        # Real W_O parameters are used here,
        # so W_O receives the soft backward signal.
        head_projected = cast_linear(
            backward_context,
            self.output,
        )

        # ==========================================================
        # COMBINE
        # ==========================================================
        #
        # Numerical forward:
        #
        #   router_projected
        #   + head_projected
        #   - head_projected
        #
        # = router_projected
        #
        # = W_O(m * C)
        #
        #
        # Gradient:
        #
        #   router_projected
        #       -> router only
        #
        #   head_projected
        #       -> Q/K/V/W_O
        #
        #   - head_projected.detach()
        #       -> cancels numerical forward only
        # ==========================================================

        context_layer = (
            router_projected
            + head_projected
            - head_projected.detach()
        )

        return context_layer

    # ==============================================================
    # Forward
    # ==============================================================

    def forward(
        self,
        hidden_states,
        attention_mask,
        router_mask,
        head_backward_mask=None,
    ):

        # ----------------------------------------------------------
        # QKV projection
        #
        # [B,S,D]
        # ->
        # [B,S,3 * total_head_dim]
        # ----------------------------------------------------------
        qkv_proj = cast_linear(
            hidden_states,
            self.qkv,
        )

        batch_size, seq_len, _ = (
            hidden_states.size()
        )

        # ----------------------------------------------------------
        # Split Q / K / V
        #
        # each:
        # [B,S,total_head_dim]
        # ----------------------------------------------------------
        q, k, v = qkv_proj.split(
            self.total_head_dim,
            dim=-1,
        )

        # ----------------------------------------------------------
        # SQK scaling
        # ----------------------------------------------------------
        sqk = (
            self.sqk
            * (
                self.ngpt_sqk_init_value
                / self.ngpt_sqk_init_scale
            )
        )

        sqk = sqk.view(
            1,
            self.num_attention_heads,
            1,
            self.d_head,
        )

        eval_backend = (
            self._eval_backend
        )

        # ----------------------------------------------------------
        # Reshape Q/K/V
        #
        # [B,S,total_head_dim]
        # ->
        # [B,S,H,d]
        # ->
        # [B,H,S,d]
        # ----------------------------------------------------------
        q = q.view(
            batch_size,
            seq_len,
            self.num_attention_heads,
            self.d_head,
        )

        k = k.view(
            batch_size,
            seq_len,
            self.num_attention_heads,
            self.d_head,
        )

        v = v.view(
            batch_size,
            seq_len,
            self.num_attention_heads,
            self.d_head,
        )

        q = q.permute(
            0,
            2,
            1,
            3,
        )

        k = k.permute(
            0,
            2,
            1,
            3,
        )

        v = v.permute(
            0,
            2,
            1,
            3,
        )

        # ==========================================================
        # TRAINING / TPU / DENSE MODE
        # ==========================================================

        if (
            self.training
            or eval_backend == "dense"
        ):

            # ------------------------------------------------------
            # Normalize q and k
            # ------------------------------------------------------
            q = justnorm(q)
            k = justnorm(k)

            # ------------------------------------------------------
            # RoPE
            # ------------------------------------------------------
            q = self.RoPE(q)
            k = self.RoPE(k)

            # ------------------------------------------------------
            # SQK scaling
            # ------------------------------------------------------
            q = (
                sqk.to(q.dtype)
                * q
            )

            k = (
                sqk.to(k.dtype)
                * k
            )

            # ------------------------------------------------------
            # Dense attention
            #
            # [B,H,S,d]
            # ------------------------------------------------------
            context_layer = (
                F.scaled_dot_product_attention(
                    q,
                    k,
                    v,
                    attn_mask=attention_mask.to(
                        q.dtype
                    ),
                    scale=math.sqrt(
                        self.d_head
                    ),
                )
            )

            # ------------------------------------------------------
            # Exclusive Attention
            # ------------------------------------------------------
            if (
                self.config
                .use_exclusive_attention
            ):
                Vn = (
                    torch.nn.functional
                    .normalize(
                        v,
                        dim=-1,
                    )
                )

                context_layer = (
                    context_layer
                    - (
                        context_layer
                        * Vn
                    ).sum(
                        dim=-1,
                        keepdim=True,
                    )
                    * Vn
                )

            # ------------------------------------------------------
            # Permanent-head jitter
            #
            # This used to occur after router masking.
            #
            # Because permanent heads always have routing value 1
            # and elastic heads do not receive this dropout, moving
            # it immediately before the HELM_7d routing operation
            # preserves the same forward behavior.
            # ------------------------------------------------------
            if (
                self.training
                and self.num_permanent_heads > 0
            ):

                permanent_heads = (
                    context_layer[
                        :,
                        :self.num_permanent_heads,
                        :,
                        :
                    ]
                )

                elastic_heads = (
                    context_layer[
                        :,
                        self.num_permanent_heads:,
                        :,
                        :
                    ]
                )

                permanent_heads = (
                    F.dropout(
                        permanent_heads,
                        p=self.config.jitter_noise,
                        training=self.training,
                    )
                )

                context_layer = torch.cat(
                    (
                        permanent_heads,
                        elastic_heads,
                    ),
                    dim=1,
                )

            # ======================================================
            # HELM_7d TRAINING ROUTING
            # ======================================================

            if (
                self.training
                and router_mask is not None
                and head_backward_mask is not None
            ):

                context_layer = (
                    self._helm7d_project(
                        context_layer=context_layer,
                        router_mask=router_mask,
                        head_backward_mask=head_backward_mask,
                        batch_size=batch_size,
                        seq_len=seq_len,
                    )
                )

            # ======================================================
            # HELM_7c-compatible dense/eval fallback
            # ======================================================
            #
            # Used for:
            #   - eval_backend == "dense"
            #   - backwards compatibility if second mask is absent
            # ======================================================

            else:

                if router_mask is not None:

                    context_layer = (
                        context_layer
                        * router_mask.expand_as(
                            context_layer
                        )
                    )

                # [B,H,S,d] -> [B,S,H,d]
                context_reshaped = (
                    context_layer
                    .permute(
                        0,
                        2,
                        1,
                        3,
                    )
                    .contiguous()
                )

                # [B,S,H,d] -> [B,S,H*d]
                context_reshaped = (
                    context_reshaped.view(
                        batch_size,
                        seq_len,
                        -1,
                    )
                )

                # 2048 -> 1024
                context_layer = cast_linear(
                    context_reshaped,
                    self.output,
                )

        # ==========================================================
        # FLEX ATTENTION
        # Batched GPU inference
        # ==========================================================

        elif (
            eval_backend == "flex"
            and batch_size > 1
        ):

            # ------------------------------------------------------
            # Normalize q and k
            # ------------------------------------------------------
            q = justnorm(q)
            k = justnorm(k)

            # ------------------------------------------------------
            # RoPE
            # ------------------------------------------------------
            q = self.RoPE(q)
            k = self.RoPE(k)

            # ------------------------------------------------------
            # SQK scaling
            # ------------------------------------------------------
            q = (
                sqk.to(q.dtype)
                * q
            )

            k = (
                sqk.to(k.dtype)
                * k
            )

            # ------------------------------------------------------
            # Hard active-head decisions
            #
            # [B,H,1,1] -> [B,H]
            # ------------------------------------------------------
            active = (
                router_mask[
                    :,
                    :,
                    0,
                    0,
                ] > 0
            )

            # ------------------------------------------------------
            # Valid keys
            #
            # [B,1,1,S] -> [B,S]
            # ------------------------------------------------------
            key_valid = (
                attention_mask[
                    :,
                    0,
                    0,
                    :
                ] >= 0
            )

            # ------------------------------------------------------
            # FlexAttention mask
            # ------------------------------------------------------
            def mask_mod(
                bi,
                hi,
                qi,
                ki,
            ):
                return (
                    active[bi, hi]
                    & key_valid[bi, ki]
                )

            block_mask = (
                self._build_block_mask(
                    mask_mod,
                    batch_size,
                    self.num_attention_heads,
                    seq_len,
                    q.device,
                )
            )

            # ------------------------------------------------------
            # Flex attention
            # ------------------------------------------------------
            context_layer = (
                self._flex_attn(
                    q,
                    k,
                    v,
                    block_mask=block_mask,
                    scale=math.sqrt(
                        self.d_head
                    ),
                )
            )

            # ------------------------------------------------------
            # Exclusive Attention
            # ------------------------------------------------------
            if (
                self.config
                .use_exclusive_attention
            ):
                Vn = (
                    torch.nn.functional
                    .normalize(
                        v,
                        dim=-1,
                    )
                )

                context_layer = (
                    context_layer
                    - (
                        context_layer
                        * Vn
                    ).sum(
                        dim=-1,
                        keepdim=True,
                    )
                    * Vn
                )

            # ------------------------------------------------------
            # Zero inactive heads.
            #
            # During inference the forward mask is hard, so
            # head_backward_mask is deliberately irrelevant.
            # ------------------------------------------------------
            context_layer = (
                context_layer
                * router_mask.expand_as(
                    context_layer
                )
            )

            # [B,H,S,d] -> [B,S,H,d]
            context_reshaped = (
                context_layer
                .permute(
                    0,
                    2,
                    1,
                    3,
                )
                .contiguous()
            )

            # -> [B,S,H*d]
            context_reshaped = (
                context_reshaped.view(
                    batch_size,
                    seq_len,
                    -1,
                )
            )

            context_layer = cast_linear(
                context_reshaped,
                self.output,
            )

        # ==========================================================
        # GATHER BACKEND
        # Single-query / batch_size == 1 inference
        # ==========================================================

        else:

            # This backend uses batch element 0's router decision,
            # so batch_size must be exactly 1.
            assert batch_size == 1, (
                "HELMSelfAttention's 'gather' eval backend "
                "only supports batch_size == 1 "
                f"(got batch_size={batch_size}); "
                "use backend='flex' for batched inference."
            )

            # ------------------------------------------------------
            # Get active heads
            # ------------------------------------------------------
            active_indices = torch.nonzero(
                router_mask[
                    0,
                    :,
                    0,
                    0,
                ]
            ).squeeze(-1)

            # ------------------------------------------------------
            # Slice Q/K/V to active heads
            # ------------------------------------------------------
            q_sliced = q[
                :,
                active_indices,
                :,
                :
            ]

            k_sliced = k[
                :,
                active_indices,
                :,
                :
            ]

            v_sliced = v[
                :,
                active_indices,
                :,
                :
            ]

            # ------------------------------------------------------
            # Normalize
            # ------------------------------------------------------
            q_sliced = justnorm(
                q_sliced
            )

            k_sliced = justnorm(
                k_sliced
            )

            # ------------------------------------------------------
            # RoPE
            # ------------------------------------------------------
            q_sliced = self.RoPE(
                q_sliced
            )

            k_sliced = self.RoPE(
                k_sliced
            )

            # ------------------------------------------------------
            # SQK scaling
            # ------------------------------------------------------
            sqk_sliced = sqk[
                :,
                active_indices,
                :,
                :
            ]

            q_sliced = (
                sqk_sliced.to(
                    q_sliced.dtype
                )
                * q_sliced
            )

            k_sliced = (
                sqk_sliced.to(
                    k_sliced.dtype
                )
                * k_sliced
            )

            # ------------------------------------------------------
            # Active-head attention
            # ------------------------------------------------------
            context_sliced = (
                F.scaled_dot_product_attention(
                    q_sliced,
                    k_sliced,
                    v_sliced,
                    attn_mask=attention_mask.to(
                        q.dtype
                    ),
                    scale=math.sqrt(
                        self.d_head
                    ),
                )
            )

            # ------------------------------------------------------
            # Exclusive Attention
            # ------------------------------------------------------
            if (
                self.config
                .use_exclusive_attention
            ):
                Vn = (
                    torch.nn.functional
                    .normalize(
                        v_sliced,
                        dim=-1,
                    )
                )

                context_sliced = (
                    context_sliced
                    - (
                        context_sliced
                        * Vn
                    ).sum(
                        dim=-1,
                        keepdim=True,
                    )
                    * Vn
                )

            # ------------------------------------------------------
            # Preserve exact router forward values.
            #
            # In HELM_7d inference these are still hard 1s for
            # active heads.
            # ------------------------------------------------------
            active_weights = (
                router_mask[
                    :,
                    active_indices,
                    :,
                    :
                ]
            )

            context_sliced = (
                context_sliced
                * active_weights
            )

            # ------------------------------------------------------
            # [1,H_active,S,d]
            # ->
            # [1,S,H_active,d]
            # ------------------------------------------------------
            context_reshaped = (
                context_sliced
                .permute(
                    0,
                    2,
                    1,
                    3,
                )
                .contiguous()
            )

            # ------------------------------------------------------
            # Flatten active heads
            # ------------------------------------------------------
            context_reshaped = (
                context_reshaped.view(
                    batch_size,
                    seq_len,
                    -1,
                )
            )

            # ------------------------------------------------------
            # Map active head IDs to their corresponding W_O columns
            #
            # head h corresponds to:
            #
            #   h*d_head
            #       ...
            #   h*d_head + d_head - 1
            # ------------------------------------------------------
            dim_offsets = torch.arange(
                self.d_head,
                device=hidden_states.device,
            )

            active_dims = (
                active_indices.unsqueeze(1)
                * self.d_head
                + dim_offsets
            ).view(-1)

            # ------------------------------------------------------
            # Slice output projection
            # ------------------------------------------------------
            sliced_weight = (
                self.output.weight[
                    :,
                    active_dims,
                ]
                .to(
                    context_reshaped.dtype
                )
            )

            sliced_bias = (
                None
                if self.output.bias is None
                else self.output.bias.to(
                    context_reshaped.dtype
                )
            )

            # ------------------------------------------------------
            # Active-only output projection
            # ------------------------------------------------------
            context_layer = F.linear(
                context_reshaped,
                sliced_weight,
                bias=sliced_bias,
            )

        # ----------------------------------------------------------
        # Return final attention residual contribution
        # ----------------------------------------------------------
        return context_layer



# HELMMLP (FFN of nGPT architecture)
# All of this stays the same from the original nGPT paper
class HELMMLP(nn.Module):

    # Define the Following:
    #   - Constants from config (for convience?)
    #       * hidden_size
    #       * ngpt_alpha_value_attn
    #       * ngpt_alpha_scale_attn
    #       * ngpt_alpha_value_mlp
    #       * ngpt_alpha_scale_mlp
    #       * ngpt_suv_value
    #       * ngpt_suv_scale
    #   - Eigen learning rate after attention (attn_alpha)
    #   - Eigen learning rate after mlp (mlp_alpha)
    #   - MLP expansion layer (mlp_exp)
    #   - suv scaling vectors for SwiGLU (suv)
    #   - SiLU() activation (silu)
    #   - MLP projection layer (mlp_expand)
    def __init__(self, config):
        super().__init__()

        # Gather Config Values for convience
        self.hidden_size = config.hidden_size
        self.ngpt_alpha_value_attn = config.ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = config.ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = config.ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = config.ngpt_alpha_scale_mlp
        self.ngpt_suv_value = config.ngpt_suv_value
        self.ngpt_suv_scale = config.ngpt_suv_scale
        self.intermediate_size = config.intermediate_size

        # Alpha Eigen Update after Attention (1st Optimizer Step)
        self.attn_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_attn*torch.ones(self.hidden_size))

        # Alpha Eigen Update after MLP (2nd Optimizer Step)
        self.mlp_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_mlp*torch.ones(self.hidden_size))

        # MLP expansion layer
        self.mlp_exp = nn.Linear(
            self.hidden_size,
            2 * self.intermediate_size,
            bias = config.bias
        )

        # suv scaling vectors during SwiGLU
        self.suv = torch.nn.Parameter(self.ngpt_suv_scale*torch.ones(2 * self.intermediate_size))

        # Define SiLU()
        self.silu = nn.SiLU()

        # MLP projection layer (shrink)
        self.mlp_proj  = nn.Linear(
            self.intermediate_size,
            self.hidden_size,
            bias=config.bias
        )

    # Peform MLP from the output of the output matrix to the end of the transformer block
    def forward(self, hidden_states, hidden_states_attention):

        # Even more convience
        hidden_size = self.hidden_size
        ngpt_alpha_value_attn = self.ngpt_alpha_value_attn
        ngpt_alpha_scale_attn = self.ngpt_alpha_scale_attn
        ngpt_alpha_value_mlp = self.ngpt_alpha_value_mlp
        ngpt_alpha_scale_mlp = self.ngpt_alpha_scale_mlp
        ngpt_suv_value = self.ngpt_suv_value
        ngpt_suv_scale = self.ngpt_suv_scale

        # Mostly Lifted from the nGPT model.py

        # Apply Normalization to hidden states before and after attention
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states)
        B_norm = justnorm(hidden_states_attention)

        # Define the eigen learning rate
        # alpha >=0
        # size(): [hidden_size]
        lr = self.attn_alpha * (ngpt_alpha_value_attn / ngpt_alpha_scale_attn)
        lr = torch.abs(lr).to(A_norm.dtype)

        # h = Norm(h + alpha_a * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt1 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt1 = justnorm(hidden_states_opt1)

        # Get u and v matrices by multiplying by mlp_exp
        # size(): [b, seq_len, hidden_size] * [hidden_size, 2 * intermediate_size] = [b, seq_len, 2 * intermediate_size]
        uv_pre = cast_linear(hidden_states_opt1 ,self.mlp_exp)
        # prepare scaling vector suv
        # size(): [intermediate_size * 2] (remember, they are concatenated)
        suv = self.suv * (ngpt_suv_value/ngpt_suv_scale) * (hidden_size ** 0.5)
        # We need to keep suv to be bf16. The line above promoted suc fp32 and the autocaster didn't fix it
        suv = suv.to(uv_pre.dtype)

        # element-wise uv by scaling vector suv
        # size(): [b, seq_len, 2 * intermediate_size]
        uv_post_suv = suv * uv_pre

        # Chunk uv into u and v
        # both size(): [b, seq_len, intermediate_size]
        u, v = torch.chunk(uv_post_suv, 2, dim=-1)

        # Apply u * silu(v), the whole point of SwiGLU (element-wise)
        # size(): [b, seq_len, intermediate_size]
        x_mlp = u * self.silu(v)

        # Project x_mlp to the mlp_proj layer (shrink)
        # size(): [b, seq_len, intermediate_size] * [intermediate_size, hidden_size] = [b, seq_len, hidden_size]
        h_mlp = cast_linear(x_mlp, self.mlp_proj)

        # Apply Normalization to hidden states after attention and after mlp
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states_opt1)
        B_norm = justnorm(h_mlp)

        # Define the eigen learning rate
        # alpha >=0
        # size(): [hidden_size]
        lr = self.mlp_alpha * (ngpt_alpha_value_mlp / ngpt_alpha_scale_mlp)
        lr = torch.abs(lr).to(A_norm.dtype)

        # h = Norm(h + alpha_m * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt2 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt2 = justnorm(hidden_states_opt2)

        # Return new hidden_state
        return hidden_states_opt2



# HELMBLOCK = HELMMultiViewRouter + HELMSelfAttention + HELMMLP
class HELMBlock(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.mlt_vw_rtr = HELMMultiViewRouter(config)
        self.attn = HELMSelfAttention(config)
        self.mlp = HELMMLP(config)

    def forward(self, hidden_states, attention_mask, easiness_score):
        router_mask, head_backward_mask = self.mlt_vw_rtr(hidden_states, easiness_score)
        count_loss = self.mlt_vw_rtr.count_loss
        attn_output = self.attn(hidden_states, attention_mask, router_mask, head_backward_mask)
        layer_output = self.mlp(hidden_states, attn_output)
        return layer_output, count_loss


class HELMModel(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.use_ckpt = config.use_ckpt
        self.embedding = HELMEmbedding(config)
        self.blocks = nn.ModuleList([HELMBlock(config) for _ in range(config.num_hidden_layers)])

    def forward(self, input_ids, attention_mask, easiness_score=None):
        attention_mask = attention_mask.unsqueeze(1).unsqueeze(2).to(torch.bfloat16)
        attention_mask = attention_mask.masked_fill(attention_mask == 0, float('-inf'))
        attention_mask = attention_mask.masked_fill(attention_mask == 1, 0.0)

        hidden_states = self.embedding(input_ids).to(torch.bfloat16)
        total_count_loss = hidden_states.new_zeros(())

        for block in self.blocks:
            if self.use_ckpt and self.training:
                _ckpt = (_xla_checkpoint if (_xla_checkpoint is not None
                         and hidden_states.device.type == "xla")
                         else torch.utils.checkpoint.checkpoint)
                hidden_states, count_loss = _ckpt(
                    block,
                    hidden_states,
                    attention_mask,
                    easiness_score,
                    use_reentrant=True if hidden_states.device.type == "xla" else False,
                )
            else:
                hidden_states, count_loss = block(hidden_states, attention_mask, easiness_score)
            total_count_loss = total_count_loss + count_loss

        # Count supervision is per-layer; average so lambda is independent of depth.
        total_count_loss = total_count_loss / float(len(self.blocks))
        return hidden_states, total_count_loss


class HELMForMaskedLM(PreTrainedModel):

    config_class = HELMConfig

    def __init__(self, config):
        super().__init__(config)
        self.ngpt_sz_init_value = config.ngpt_sz_init_value
        self.ngpt_sz_init_scale = config.ngpt_sz_init_scale
        self.model = HELMModel(config)
        self.classifier = nn.Linear(config.hidden_size, config.vocab_size, bias=config.bias)
        self.sz = nn.Parameter(torch.ones(config.vocab_size))
        self.post_init()

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)

    def enable_efficient_inference(self, backend="flex", compile=True):
        for block in self.model.blocks:
            block.attn.set_eval_backend(backend=backend, compile=compile)
        return self

    @torch.no_grad()
    def normalize_ngpt_matrices(self):
        # q_up_proj is intentionally EXCLUDED: HELM_7c allows router magnitude.
        keys_to_normalize = (
            "word_embeddings.weight",
            "classifier.weight",
            "attn.qkv.weight",
            "attn.output.weight",
            "mlp.mlp_exp.weight",
            "mlp.mlp_proj.weight",
            "mlt_vw_rtr.q_down_proj.weight",
        )
        for name, param in self.named_parameters():
            if name.endswith(keys_to_normalize):
                param.copy_(justnorm(param, dim=1, eps=1e-12))

    @torch.no_grad()
    def get_telemetry(self):
        telemetry = {}

        for i, block in enumerate(self.model.blocks):
            router = block.mlt_vw_rtr
            logits = router.save_router_logits.float().cpu()
            sigmoid = router.save_sigmoid_scores.float().cpu()
            hard = router.save_hard_mask.float().cpu()
            actual = router.save_total_head_count.float().cpu()
            target = router.save_target_total_head_count.float().cpu()
            error = router.save_count_error.float().cpu()
            q_up_norms = router.q_up_proj.weight.detach().float().norm(dim=1).cpu()

            telemetry[f"layer_{i}_router_logits"] = logits
            telemetry[f"layer_{i}_sigmoid_scores"] = sigmoid
            telemetry[f"layer_{i}_hard_mask"] = hard
            telemetry[f"layer_{i}_elastic_head_ratio"] = hard.mean().item()
            telemetry[f"layer_{i}_total_head_count_mean"] = actual.mean().item()
            telemetry[f"layer_{i}_target_head_count_mean"] = target.mean().item()
            telemetry[f"layer_{i}_count_error_mean"] = error.mean().item()
            telemetry[f"layer_{i}_count_error_mae"] = error.abs().mean().item()
            telemetry[f"layer_{i}_count_loss"] = router.save_count_loss.float().item()
            telemetry[f"layer_{i}_router_weight_norms"] = q_up_norms
            telemetry[f"layer_{i}_router_weight_norm_mean"] = q_up_norms.mean().item()
            telemetry[f"layer_{i}_router_weight_norm_std"] = q_up_norms.std(unbiased=False).item()
            telemetry[f"layer_{i}_l_i_weights"] = router.l_i_weights.detach().float().cpu()

        return telemetry

    def forward(self, input_ids, attention_mask, current_step=None, easiness_score=None):
        # current_step is accepted only for backward compatibility with older callers.
        features, total_count_loss = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            easiness_score=easiness_score,
        )

        sz = self.sz * (self.ngpt_sz_init_value / self.ngpt_sz_init_scale)
        unscaled_logits = cast_linear(features, self.classifier)
        logits = sz.to(unscaled_logits.dtype) * unscaled_logits
        return logits, total_count_loss



Writing model.py


In [8]:
%%writefile analyze_helm7c_heads.py
"""
HELM_7d focused checkpoint diagnostic.

Purpose
-------
HELM_7d changed only the backward training path of inactive elastic heads.
The forward pass is still hard-routed.

This diagnostic therefore focuses on four questions:

1) Did routing/cardinality drift upward?
   - actual vs target head count
   - count error / count loss
   - sigmoid distribution and near-threshold concentration
   - fraction of elastic heads ON
   - router weight norms

2) How strong is the new inactive-head rescue signal?
   - mean backward score for OFF heads
   - fraction of OFF decisions receiving >0.25 / >0.45 gradient strength

3) Did the head bank become healthier?
   - per-head residual RMS / energy
   - effective rank for all / permanent / elastic heads
   - activation-frequency vs residual-RMS Spearman correlation

4) Did the router still learn meaningful head identities?
   - routed CE
   - forced-dense CE
   - permanent-only CE
   - random-same-count CE

Removed on purpose
------------------
- signed-gradient "oracle"
- temporary-gate utility analysis
- exact single-head ablation
- Q/K/V/O parameter cosine
- pairwise functional-cosine analysis

Those were useful for HELM_7c, but they do not directly test the HELM_7d
starvation-rescue hypothesis and make the script much more complicated.

Default checkpoint:
  repo: JamesResearch1216/HELM_7d
  file: checkpoint-006500.pt
"""

from __future__ import annotations

import argparse
import contextlib
import csv
import glob
import json
import math
import os
import random
import shutil
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

try:
    from huggingface_hub import hf_hub_download
except Exception as exc:
    raise RuntimeError(
        "huggingface_hub is required. Install it with: pip install huggingface_hub"
    ) from exc

try:
    import pyarrow.parquet as pq
except Exception as exc:
    raise RuntimeError(
        "pyarrow is required. Install it with: pip install pyarrow"
    ) from exc

SCRIPT_DIR = Path.cwd()
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

try:
    from model import HELMConfig, HELMForMaskedLM, justnorm, cast_linear
except Exception as exc:
    raise RuntimeError(
        "Could not import HELM_7d model.py. Run the notebook model.py cell first."
    ) from exc


# -----------------------------------------------------------------------------
# Defaults
# -----------------------------------------------------------------------------
MODEL_REPO = "JamesResearch1216/HELM_7d"
CHECKPOINT_FILE = "checkpoint-006500.pt"
TRAINING_STATE_FILE = "training_state.json"
DATA_REPO = "JamesResearch1216/HELM-Easiness-Data-10B-Labeled-v6"
VALIDATION_FILE = "data/seq_1024/validation-00000.parquet"


# -----------------------------------------------------------------------------
# Generic helpers
# -----------------------------------------------------------------------------
def get_hf_token() -> Optional[str]:
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def strip_state_prefixes(state: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    out = {}
    for key, value in state.items():
        k = key
        changed = True
        while changed:
            changed = False
            for prefix in ("module.", "_orig_mod."):
                if k.startswith(prefix):
                    k = k[len(prefix):]
                    changed = True
        out[k] = value
    return out


def double_argsort_rank(x: torch.Tensor, descending: bool = True) -> torch.Tensor:
    order = torch.argsort(x, dim=-1, descending=descending)
    return torch.argsort(order, dim=-1)


def rankdata_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty(len(x), dtype=np.float64)
    sorted_x = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sorted_x[j] == sorted_x[i]:
            j += 1
        avg_rank = 0.5 * (i + j - 1)
        ranks[order[i:j]] = avg_rank
        i = j
    return ranks


def spearman_np(x, y) -> float:
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    good = np.isfinite(x) & np.isfinite(y)
    x = x[good]
    y = y[good]
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return float("nan")
    return float(np.corrcoef(rankdata_np(x), rankdata_np(y))[0, 1])


def effective_rank_from_gram(gram: np.ndarray) -> Tuple[float, float, np.ndarray]:
    g = 0.5 * (gram + gram.T)
    vals = np.linalg.eigvalsh(g)
    vals = np.clip(vals, 0.0, None)
    total = vals.sum()
    if total <= 1e-12:
        return 0.0, 0.0, vals
    p = vals / total
    p_pos = p[p > 1e-12]
    entropy_rank = float(np.exp(-(p_pos * np.log(p_pos)).sum()))
    participation_rank = float(
        (total * total) / (np.square(vals).sum() + 1e-12)
    )
    return entropy_rank, participation_rank, vals


def save_heatmap(
    path: Path,
    matrix: np.ndarray,
    title: str,
    xlabel: str,
    ylabel: str,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
) -> None:
    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(
        matrix,
        aspect="auto",
        interpolation="nearest",
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def save_bar(path: Path, values: np.ndarray, title: str, ylabel: str) -> None:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(np.arange(len(values)), values)
    ax.set_title(title)
    ax.set_xlabel("Head")
    ax.set_ylabel(ylabel)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def save_hist(path: Path, values: np.ndarray, title: str, xlabel: str) -> None:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(np.asarray(values).reshape(-1), bins=30)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


# -----------------------------------------------------------------------------
# Device
# -----------------------------------------------------------------------------
@dataclass
class DeviceContext:
    device: torch.device
    kind: str
    xm: object = None
    autocast_dtype: Optional[torch.dtype] = None

    def autocast(self):
        if self.kind == "cuda":
            return torch.autocast(
                device_type="cuda",
                dtype=self.autocast_dtype,
            )
        if self.kind == "xla":
            try:
                return torch.autocast(
                    device_type="xla",
                    dtype=torch.bfloat16,
                )
            except Exception:
                return contextlib.nullcontext()
        return contextlib.nullcontext()

    def mark_step(self):
        if self.kind == "xla" and self.xm is not None:
            self.xm.mark_step()


def resolve_device(requested: str) -> DeviceContext:
    requested = requested.lower()

    if requested == "auto":
        if torch.cuda.is_available():
            requested = "cuda"
        elif glob.glob("/dev/accel*"):
            requested = "xla"
        else:
            requested = "cpu"

    if requested == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("--device cuda requested but CUDA is unavailable")
        dtype = (
            torch.bfloat16
            if torch.cuda.is_bf16_supported()
            else torch.float16
        )
        return DeviceContext(
            torch.device("cuda"),
            "cuda",
            autocast_dtype=dtype,
        )

    if requested == "xla":
        try:
            import torch_xla.core.xla_model as xm
        except Exception as exc:
            raise RuntimeError(
                "--device xla requested but torch_xla could not be imported. "
                "Run/restart with your normal TPU setup first."
            ) from exc

        return DeviceContext(
            xm.xla_device(),
            "xla",
            xm=xm,
            autocast_dtype=torch.bfloat16,
        )

    if requested == "cpu":
        return DeviceContext(
            torch.device("cpu"),
            "cpu",
        )

    raise ValueError(f"Unknown device: {requested}")


# -----------------------------------------------------------------------------
# Download / load
# -----------------------------------------------------------------------------
def download_assets(
    cache_dir: Path,
    token: Optional[str],
    model_repo: str,
    checkpoint_file: str,
    data_repo: str,
    validation_file: str,
):
    cache_dir.mkdir(parents=True, exist_ok=True)

    print(f"Downloading checkpoint: {model_repo}/{checkpoint_file}")
    ckpt_path = hf_hub_download(
        repo_id=model_repo,
        filename=checkpoint_file,
        repo_type="model",
        token=token,
        local_dir=str(cache_dir / "model_repo"),
    )

    training_state_path = None
    try:
        training_state_path = hf_hub_download(
            repo_id=model_repo,
            filename=TRAINING_STATE_FILE,
            repo_type="model",
            token=token,
            local_dir=str(cache_dir / "model_repo"),
        )
    except Exception as exc:
        print(f"WARNING: could not download training_state.json: {exc}")

    print(f"Downloading validation shard: {data_repo}/{validation_file}")
    validation_path = hf_hub_download(
        repo_id=data_repo,
        filename=validation_file,
        repo_type="dataset",
        token=token,
        local_dir=str(cache_dir / "dataset"),
    )

    return (
        Path(ckpt_path),
        Path(training_state_path) if training_state_path else None,
        Path(validation_path),
    )


def load_breakpoints(training_state_path: Optional[Path]) -> Optional[List[float]]:
    if training_state_path is None or not training_state_path.exists():
        return None

    try:
        with training_state_path.open("r") as f:
            state = json.load(f)

        easiness_dict = state.get("easiness_dict")
        if isinstance(easiness_dict, dict):
            bp = easiness_dict.get("breakpoints")
            if bp:
                print(
                    f"Loaded {len(bp)} easiness CDF breakpoints "
                    "from training_state.json"
                )
                return bp

    except Exception as exc:
        print(f"WARNING: failed to parse easiness breakpoints: {exc}")

    return None


def load_model(
    ckpt_path: Path,
    breakpoints: Optional[List[float]],
    dev: DeviceContext,
):
    config = HELMConfig(
        easiness_cdf_breakpoints=breakpoints
    )

    model = HELMForMaskedLM(config)

    print(f"Loading checkpoint from {ckpt_path}")
    payload = torch.load(
        str(ckpt_path),
        map_location="cpu",
    )

    if not isinstance(payload, dict):
        raise RuntimeError("Checkpoint is not a dict")

    state = strip_state_prefixes(
        payload.get("model_state", payload)
    )

    missing, unexpected = model.load_state_dict(
        state,
        strict=False,
    )

    if missing or unexpected:
        print(
            f"State load: {len(missing)} missing keys, "
            f"{len(unexpected)} unexpected keys"
        )
        if missing:
            print("  Missing (first 10):", missing[:10])
        if unexpected:
            print("  Unexpected (first 10):", unexpected[:10])

        if len(missing) > 5 or len(unexpected) > 5:
            raise RuntimeError(
                "Large state-dict mismatch. Make sure this notebook's "
                "model.py matches the HELM_7d checkpoint."
            )

    del payload

    model.to(dev.device)
    model.eval()

    # Analysis deliberately uses compute-all-then-mask, matching training shapes.
    model.enable_efficient_inference(
        "dense",
        compile=False,
    )

    return model, config


# -----------------------------------------------------------------------------
# Deterministic validation masking
# -----------------------------------------------------------------------------
def deterministic_span_mask(
    ids: torch.Tensor,
    config: HELMConfig,
    seed: int,
    probability: float = 0.30,
    span_length: int = 3,
) -> Tuple[torch.Tensor, torch.Tensor]:

    ids = ids.clone().long()
    labels = torch.full_like(
        ids,
        -100,
    )

    g = torch.Generator(
        device="cpu"
    )
    g.manual_seed(
        int(seed)
    )

    special = {
        int(config.bos_token_id),
        int(config.eos_token_id),
        int(config.pad_token_id),
        int(config.mask_token_id),
        int(config.unk_token_id),
    }

    candidate = [
        i
        for i, tok in enumerate(ids.tolist())
        if int(tok) not in special
    ]

    if not candidate:
        return ids, labels

    target = max(
        1,
        int(round(probability * len(candidate))),
    )

    candidate_set = set(candidate)
    perm = torch.randperm(
        len(candidate),
        generator=g,
    ).tolist()

    chosen = set()

    for pi in perm:
        if len(chosen) >= target:
            break

        start = candidate[pi]

        for pos in range(
            start,
            min(
                start + span_length,
                ids.numel(),
            ),
        ):
            if pos in candidate_set:
                chosen.add(pos)
                if len(chosen) >= target:
                    break

    chosen = sorted(chosen)

    if not chosen:
        chosen = [candidate[0]]

    pos = torch.tensor(
        chosen,
        dtype=torch.long,
    )

    original = ids[pos].clone()
    labels[pos] = original

    r = torch.rand(
        len(pos),
        generator=g,
    )

    mask_sel = r < 0.80
    random_sel = (
        (r >= 0.80)
        & (r < 0.90)
    )

    ids[pos[mask_sel]] = int(
        config.mask_token_id
    )

    if random_sel.any():
        random_tokens = torch.randint(
            low=0,
            high=int(config.vocab_size),
            size=(int(random_sel.sum()),),
            generator=g,
        )

        ids[pos[random_sel]] = random_tokens

    return ids, labels


def prepare_batches(
    validation_path: Path,
    config: HELMConfig,
    num_examples: int,
    batch_size: int,
    seq_len: int,
    seed: int,
) -> List[Dict[str, torch.Tensor]]:

    table = pq.read_table(
        str(validation_path),
        columns=[
            "input_ids",
            "easiness_score",
        ],
    )

    total_rows = table.num_rows
    n = min(
        num_examples,
        total_rows,
    )

    rng = np.random.default_rng(
        seed
    )

    indices = rng.permutation(
        total_rows
    )[:n]

    input_col = table.column(
        "input_ids"
    )
    easy_col = table.column(
        "easiness_score"
    )

    examples = []

    for i, row_idx in enumerate(
        indices.tolist()
    ):
        ids_list = input_col[
            row_idx
        ].as_py()

        easy_val = easy_col[
            row_idx
        ].as_py()

        ids = torch.tensor(
            ids_list,
            dtype=torch.long,
        )[:seq_len]

        if ids.numel() < seq_len:
            pad = torch.full(
                (
                    seq_len
                    - ids.numel(),
                ),
                int(config.pad_token_id),
                dtype=torch.long,
            )

            ids = torch.cat(
                [
                    ids,
                    pad,
                ],
                dim=0,
            )

        masked, labels = deterministic_span_mask(
            ids,
            config,
            seed=seed + 100003 * i,
        )

        examples.append(
            {
                "input_ids": masked,
                "labels": labels,
                "attention_mask": (
                    masked
                    != int(
                        config.pad_token_id
                    )
                ).long(),
                "easiness_score": torch.tensor(
                    float(easy_val),
                    dtype=torch.float32,
                ),
                "example_id": torch.tensor(
                    i,
                    dtype=torch.long,
                ),
            }
        )

    usable = (
        len(examples)
        // batch_size
    ) * batch_size

    examples = examples[:usable]

    if not examples:
        raise RuntimeError(
            "Not enough examples for one complete batch"
        )

    batches = []

    for start in range(
        0,
        len(examples),
        batch_size,
    ):
        chunk = examples[
            start:start + batch_size
        ]

        batches.append(
            {
                k: torch.stack(
                    [
                        x[k]
                        for x in chunk
                    ],
                    dim=0,
                )
                for k in chunk[0]
            }
        )

    print(
        f"Prepared {len(examples)} examples -> "
        f"{len(batches)} batches of {batch_size}, "
        f"seq_len={seq_len}"
    )

    return batches


def move_batch(
    batch: Dict[str, torch.Tensor],
    dev: DeviceContext,
) -> Dict[str, torch.Tensor]:
    return {
        k: v.to(dev.device)
        for k, v in batch.items()
    }


# -----------------------------------------------------------------------------
# CE
# -----------------------------------------------------------------------------
def ce_sum_and_count(
    logits: torch.Tensor,
    labels: torch.Tensor,
    chunk_tokens: int = 128,
):
    total = logits.new_zeros(
        (),
        dtype=torch.float32,
    )

    count = labels.new_zeros(
        (),
        dtype=torch.long,
    )

    seq_len = logits.size(1)
    vocab = logits.size(-1)

    for start in range(
        0,
        seq_len,
        chunk_tokens,
    ):
        end = min(
            start + chunk_tokens,
            seq_len,
        )

        lgt = (
            logits[
                :,
                start:end,
                :
            ]
            .float()
            .reshape(
                -1,
                vocab,
            )
        )

        lab = labels[
            :,
            start:end
        ].reshape(-1)

        total = total + F.cross_entropy(
            lgt,
            lab,
            ignore_index=-100,
            reduction="sum",
        )

        count = count + (
            lab != -100
        ).sum()

    return total, count


def forward_ce(
    model,
    batch: Dict[str, torch.Tensor],
    dev: DeviceContext,
    pass_easiness: bool = True,
):
    kwargs = {
        "input_ids": batch["input_ids"],
        "attention_mask": batch["attention_mask"],
        "current_step": 6500,
        "easiness_score": (
            batch["easiness_score"]
            if pass_easiness
            else None
        ),
    }

    with dev.autocast():
        logits, count_loss = model(
            **kwargs
        )

        ce_sum, ce_count = (
            ce_sum_and_count(
                logits,
                batch["labels"],
            )
        )

        ce = (
            ce_sum
            / ce_count.clamp_min(1).to(
                ce_sum.dtype
            )
        )

    return (
        ce,
        ce_sum,
        ce_count,
        count_loss,
    )


# -----------------------------------------------------------------------------
# 7d-aware router override
# -----------------------------------------------------------------------------
class RouterOverride:
    """
    HELM_7d router returns:
        (router_mask, head_backward_mask)

    In model.eval(), HELMSelfAttention uses the hard router_mask for forward
    routing and ignores the backward-only mask. We still return a matching
    second mask so the tuple architecture remains valid.
    """

    def __init__(
        self,
        model,
        mode: str,
        permanent_heads: int,
    ):
        self.model = model
        self.mode = mode
        self.permanent_heads = permanent_heads
        self.handles = []

    def _hook(
        self,
        layer_idx: int,
    ):
        def hook(
            module,
            inputs,
            output,
        ):
            if not (
                isinstance(output, tuple)
                and len(output) == 2
            ):
                raise RuntimeError(
                    "Expected HELM_7d router to return "
                    "(router_mask, head_backward_mask)"
                )

            router_mask, head_backward_mask = output

            b, h, _, _ = (
                router_mask.shape
            )
            p = self.permanent_heads

            if self.mode == "dense":
                new_mask = torch.ones_like(
                    router_mask
                )

            elif self.mode == "permanent_only":
                new_mask = torch.zeros_like(
                    router_mask
                )

                new_mask[
                    :,
                    :p,
                    :,
                    :
                ] = 1.0

            elif self.mode == "random_same_count":
                elastic = router_mask[
                    :,
                    p:,
                    0,
                    0,
                ]

                k = (
                    elastic > 0.5
                ).sum(
                    dim=-1,
                    keepdim=True,
                )

                noise = torch.rand_like(
                    elastic.float()
                )

                ranks = double_argsort_rank(
                    noise,
                    descending=True,
                )

                rand_elastic = (
                    ranks < k
                ).to(
                    router_mask.dtype
                )

                permanent = torch.ones(
                    (b, p),
                    device=router_mask.device,
                    dtype=router_mask.dtype,
                )

                new_mask = torch.cat(
                    [
                        permanent,
                        rand_elastic,
                    ],
                    dim=-1,
                ).view(
                    b,
                    h,
                    1,
                    1,
                )

            else:
                raise ValueError(
                    f"Unknown override mode: {self.mode}"
                )

            # Eval ignores the backward mask, but keeping both masks coherent
            # makes the hook architecture-safe.
            return (
                new_mask,
                new_mask.detach(),
            )

        return hook

    def __enter__(self):
        for i, block in enumerate(
            self.model.model.blocks
        ):
            self.handles.append(
                block.mlt_vw_rtr.register_forward_hook(
                    self._hook(i)
                )
            )
        return self

    def __exit__(
        self,
        exc_type,
        exc,
        tb,
    ):
        for handle in self.handles:
            handle.remove()

        self.handles.clear()


# -----------------------------------------------------------------------------
# Router/cardinality + 7d backward-rescue statistics
# -----------------------------------------------------------------------------
def collect_router_statistics(
    model,
    batches,
    dev: DeviceContext,
    output_dir: Path,
    selected_layers: Sequence[int],
):
    n_layers = len(
        model.model.blocks
    )

    logits_by_layer = [
        []
        for _ in range(n_layers)
    ]

    sig_by_layer = [
        []
        for _ in range(n_layers)
    ]

    mask_by_layer = [
        []
        for _ in range(n_layers)
    ]

    backward_by_layer = [
        []
        for _ in range(n_layers)
    ]

    actual_by_layer = [
        []
        for _ in range(n_layers)
    ]

    target_by_layer = [
        []
        for _ in range(n_layers)
    ]

    layer_count_loss = [
        []
        for _ in range(n_layers)
    ]

    easiness_all = []

    ce_total = 0.0
    ce_tokens = 0

    model_count_losses = []

    with torch.no_grad():
        for cpu_batch in batches:
            batch = move_batch(
                cpu_batch,
                dev,
            )

            (
                ce,
                ce_sum,
                ce_count,
                count_loss,
            ) = forward_ce(
                model,
                batch,
                dev,
                pass_easiness=True,
            )

            dev.mark_step()

            ce_total += float(
                ce_sum.detach().cpu()
            )

            ce_tokens += int(
                ce_count.detach().cpu()
            )

            model_count_losses.append(
                float(
                    count_loss.detach().float().cpu()
                )
            )

            easiness_all.append(
                cpu_batch[
                    "easiness_score"
                ].numpy()
            )

            for li, block in enumerate(
                model.model.blocks
            ):
                r = block.mlt_vw_rtr

                logits_by_layer[li].append(
                    r.save_router_logits
                    .detach()
                    .float()
                    .cpu()
                    .numpy()
                )

                sig_by_layer[li].append(
                    r.save_sigmoid_scores
                    .detach()
                    .float()
                    .cpu()
                    .numpy()
                )

                mask_by_layer[li].append(
                    r.save_hard_mask
                    .detach()
                    .float()
                    .cpu()
                    .numpy()
                )

                backward_by_layer[li].append(
                    r.save_head_backward_scores
                    .detach()
                    .float()
                    .cpu()
                    .numpy()
                )

                actual_by_layer[li].append(
                    r.save_total_head_count
                    .detach()
                    .float()
                    .cpu()
                    .numpy()
                )

                target_by_layer[li].append(
                    r.save_target_total_head_count
                    .detach()
                    .float()
                    .cpu()
                    .numpy()
                )

                layer_count_loss[li].append(
                    float(
                        r.save_count_loss
                        .detach()
                        .float()
                        .cpu()
                    )
                )

    easiness = np.concatenate(
        easiness_all,
        axis=0,
    )

    per_layer = {}
    calibration_rows = []

    all_logits = []
    all_sig = []
    all_masks = []
    all_backward = []
    all_actual = []
    all_target = []

    for li in range(n_layers):
        logits = np.concatenate(
            logits_by_layer[li],
            axis=0,
        )

        sig = np.concatenate(
            sig_by_layer[li],
            axis=0,
        )

        mask = np.concatenate(
            mask_by_layer[li],
            axis=0,
        )

        backward = np.concatenate(
            backward_by_layer[li],
            axis=0,
        )

        actual = np.concatenate(
            actual_by_layer[li],
            axis=0,
        )

        target = np.concatenate(
            target_by_layer[li],
            axis=0,
        )

        freq = mask.mean(
            axis=0
        )

        weight_norms = (
            model.model.blocks[
                li
            ]
            .mlt_vw_rtr
            .q_up_proj
            .weight
            .detach()
            .float()
            .norm(
                dim=1
            )
            .cpu()
            .numpy()
        )

        off = mask < 0.5
        on = mask > 0.5

        off_backward = backward[
            off
        ]

        on_backward = backward[
            on
        ]

        stats = {
            "actual_mean":
                float(actual.mean()),

            "target_mean":
                float(target.mean()),

            "count_error_mean":
                float(
                    (
                        actual
                        - target
                    ).mean()
                ),

            "count_mae":
                float(
                    np.abs(
                        actual
                        - target
                    ).mean()
                ),

            "layer_count_loss_mean":
                float(
                    np.mean(
                        layer_count_loss[
                            li
                        ]
                    )
                ),

            "easiness_actual_spearman":
                spearman_np(
                    easiness,
                    actual,
                ),

            "target_actual_spearman":
                spearman_np(
                    target,
                    actual,
                ),

            "elastic_on_fraction":
                float(
                    mask.mean()
                ),

            "dynamic_head_fraction_05_95":
                float(
                    (
                        (freq > 0.05)
                        & (freq < 0.95)
                    ).mean()
                ),

            "always_off_fraction_lt_01":
                float(
                    (freq < 0.01).mean()
                ),

            "always_on_fraction_gt_99":
                float(
                    (freq > 0.99).mean()
                ),

            "sigmoid_mean":
                float(sig.mean()),

            "sigmoid_min":
                float(sig.min()),

            "sigmoid_max":
                float(sig.max()),

            "sigmoid_below_45":
                float(
                    (sig < 0.45).mean()
                ),

            "sigmoid_near_half_45_55":
                float(
                    (
                        (sig >= 0.45)
                        & (sig <= 0.55)
                    ).mean()
                ),

            "sigmoid_above_55":
                float(
                    (sig > 0.55).mean()
                ),

            "sigmoid_saturation_lt05_gt95":
                float(
                    (
                        (sig < 0.05)
                        | (sig > 0.95)
                    ).mean()
                ),

            "ste_derivative_mean":
                float(
                    (
                        sig
                        * (1.0 - sig)
                    ).mean()
                ),

            "router_logit_mean":
                float(
                    logits.mean()
                ),

            "router_logit_std":
                float(
                    logits.std()
                ),

            "router_weight_norm_mean":
                float(
                    weight_norms.mean()
                ),

            "router_weight_norm_std":
                float(
                    weight_norms.std()
                ),

            # HELM_7d-specific backward rescue metrics
            "off_head_backward_mean":
                (
                    float(
                        off_backward.mean()
                    )
                    if off_backward.size
                    else float("nan")
                ),

            "off_head_backward_median":
                (
                    float(
                        np.median(
                            off_backward
                        )
                    )
                    if off_backward.size
                    else float("nan")
                ),

            "off_head_backward_gt_25":
                (
                    float(
                        (
                            off_backward
                            > 0.25
                        ).mean()
                    )
                    if off_backward.size
                    else float("nan")
                ),

            "off_head_backward_gt_45":
                (
                    float(
                        (
                            off_backward
                            > 0.45
                        ).mean()
                    )
                    if off_backward.size
                    else float("nan")
                ),

            "on_head_backward_mean":
                (
                    float(
                        on_backward.mean()
                    )
                    if on_backward.size
                    else float("nan")
                ),

            "min_total_head_fraction":
                float(
                    (
                        actual
                        == model.config.head_target_min
                    ).mean()
                ),

            "max_total_head_fraction":
                float(
                    (
                        actual
                        == model.config.head_target_max
                    ).mean()
                ),
        }

        per_layer[li] = stats

        head_rows = []

        for h in range(
            mask.shape[1]
        ):
            off_h = (
                mask[:, h] < 0.5
            )

            off_bwd_h = (
                backward[
                    off_h,
                    h,
                ]
            )

            head_rows.append(
                {
                    "layer": li,
                    "elastic_head": h,
                    "absolute_head":
                        h
                        + model.config.num_permanent_heads,

                    "activation_frequency":
                        float(freq[h]),

                    "mean_logit":
                        float(
                            logits[
                                :,
                                h,
                            ].mean()
                        ),

                    "std_logit":
                        float(
                            logits[
                                :,
                                h,
                            ].std()
                        ),

                    "mean_sigmoid":
                        float(
                            sig[
                                :,
                                h,
                            ].mean()
                        ),

                    "mean_backward_score":
                        float(
                            backward[
                                :,
                                h,
                            ].mean()
                        ),

                    "mean_off_backward_score":
                        (
                            float(
                                off_bwd_h.mean()
                            )
                            if off_bwd_h.size
                            else float("nan")
                        ),

                    "router_weight_norm":
                        float(
                            weight_norms[h]
                        ),
                }
            )

        with (
            output_dir
            / f"layer_{li:02d}_router_heads.csv"
        ).open(
            "w",
            newline="",
        ) as f:
            writer = csv.DictWriter(
                f,
                fieldnames=head_rows[0].keys(),
            )

            writer.writeheader()
            writer.writerows(
                head_rows
            )

        # Count calibration by integer target.
        for t in range(
            int(model.config.head_target_min),
            int(model.config.head_target_max) + 1,
        ):
            idx = target == t

            if idx.any():
                calibration_rows.append(
                    {
                        "layer": li,
                        "target_heads": t,
                        "n": int(
                            idx.sum()
                        ),
                        "actual_mean":
                            float(
                                actual[
                                    idx
                                ].mean()
                            ),
                        "actual_std":
                            float(
                                actual[
                                    idx
                                ].std()
                            ),
                        "mae":
                            float(
                                np.abs(
                                    actual[
                                        idx
                                    ]
                                    - target[
                                        idx
                                    ]
                                ).mean()
                            ),
                        "fraction_exact":
                            float(
                                (
                                    actual[
                                        idx
                                    ]
                                    == target[
                                        idx
                                    ]
                                ).mean()
                            ),
                    }
                )

        if li in selected_layers:
            save_heatmap(
                output_dir
                / f"layer_{li:02d}_hard_mask.png",
                mask,
                f"Layer {li}: elastic hard activations",
                "Elastic head",
                "Example",
                0.0,
                1.0,
            )

            save_heatmap(
                output_dir
                / f"layer_{li:02d}_sigmoid_scores.png",
                sig,
                f"Layer {li}: elastic sigmoid scores",
                "Elastic head",
                "Example",
                0.0,
                1.0,
            )

            save_heatmap(
                output_dir
                / f"layer_{li:02d}_backward_scores.png",
                backward,
                f"Layer {li}: 7d head backward strengths",
                "Elastic head",
                "Example",
                0.0,
                1.0,
            )

            save_hist(
                output_dir
                / f"layer_{li:02d}_sigmoid_hist.png",
                sig,
                f"Layer {li}: sigmoid distribution",
                "sigmoid(router logit)",
            )

            save_bar(
                output_dir
                / f"layer_{li:02d}_activation_frequency.png",
                freq,
                f"Layer {li}: elastic activation frequency",
                "Activation frequency",
            )

            save_bar(
                output_dir
                / f"layer_{li:02d}_router_weight_norms.png",
                weight_norms,
                f"Layer {li}: router row norms",
                "L2 norm",
            )

        all_logits.append(
            logits
        )
        all_sig.append(
            sig
        )
        all_masks.append(
            mask
        )
        all_backward.append(
            backward
        )
        all_actual.append(
            actual
        )
        all_target.append(
            target
        )

    if calibration_rows:
        with (
            output_dir
            / "count_calibration.csv"
        ).open(
            "w",
            newline="",
        ) as f:
            writer = csv.DictWriter(
                f,
                fieldnames=calibration_rows[0].keys(),
            )

            writer.writeheader()
            writer.writerows(
                calibration_rows
            )

    layer_rows = []

    for li, stats in per_layer.items():
        layer_rows.append(
            {
                "layer": li,
                **stats,
            }
        )

    with (
        output_dir
        / "router_layer_summary.csv"
    ).open(
        "w",
        newline="",
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=layer_rows[0].keys(),
        )

        writer.writeheader()
        writer.writerows(
            layer_rows
        )

    return {
        "routed_ce":
            ce_total
            / max(
                1,
                ce_tokens,
            ),

        "model_count_loss_mean":
            float(
                np.mean(
                    model_count_losses
                )
            ),

        "easiness":
            easiness,

        "per_layer":
            per_layer,

        "logits":
            all_logits,

        "sigmoid":
            all_sig,

        "masks":
            all_masks,

        "backward_scores":
            all_backward,

        "actual":
            all_actual,

        "target":
            all_target,
    }


# -----------------------------------------------------------------------------
# CE mode comparisons
# -----------------------------------------------------------------------------
def evaluate_override_mode(
    model,
    batches,
    dev: DeviceContext,
    mode: str,
    random_seed: int = 0,
) -> float:
    total_sum = 0.0
    total_count = 0

    seed_everything(
        random_seed
    )

    manager = (
        contextlib.nullcontext()
        if mode == "routed"
        else RouterOverride(
            model,
            mode,
            model.config.num_permanent_heads,
        )
    )

    with manager:
        with torch.no_grad():
            for cpu_batch in batches:
                batch = move_batch(
                    cpu_batch,
                    dev,
                )

                (
                    _,
                    ce_sum,
                    ce_count,
                    _,
                ) = forward_ce(
                    model,
                    batch,
                    dev,
                    pass_easiness=False,
                )

                dev.mark_step()

                total_sum += float(
                    ce_sum.detach().cpu()
                )

                total_count += int(
                    ce_count.detach().cpu()
                )

    return (
        total_sum
        / max(
            1,
            total_count,
        )
    )


# -----------------------------------------------------------------------------
# Head residual dynamics
# -----------------------------------------------------------------------------
class AttentionInputCapture:
    """
    Capture only the first two HELMSelfAttention inputs.

    HELM_7d attention signature:
        (hidden_states, attention_mask, router_mask, head_backward_mask)
    """

    def __init__(
        self,
        model,
        layers: Sequence[int],
    ):
        self.model = model
        self.layers = set(
            layers
        )
        self.handles = []
        self.data = {}

    def _hook(
        self,
        li: int,
    ):
        def hook(
            module,
            inputs,
        ):
            if len(inputs) < 2:
                raise RuntimeError(
                    "Unexpected HELMSelfAttention input signature"
                )

            hidden_states = inputs[0]
            attention_mask = inputs[1]

            self.data[li] = (
                hidden_states.detach(),
                attention_mask.detach(),
            )

        return hook

    def __enter__(self):
        for li in self.layers:
            self.handles.append(
                self.model.model.blocks[
                    li
                ].attn.register_forward_pre_hook(
                    self._hook(li)
                )
            )
        return self

    def __exit__(
        self,
        exc_type,
        exc,
        tb,
    ):
        for h in self.handles:
            h.remove()

        self.handles.clear()


def unmasked_attention_context(
    attn,
    hidden_states: torch.Tensor,
    attention_mask: torch.Tensor,
):
    """
    Recompute all head contexts before hard routing.

    This intentionally measures what each head is capable of producing,
    not only what survived the router.
    """

    qkv_proj = cast_linear(
        hidden_states,
        attn.qkv,
    )

    b, s, _ = hidden_states.shape

    q, k, v = qkv_proj.split(
        attn.total_head_dim,
        dim=-1,
    )

    q = (
        q.view(
            b,
            s,
            attn.num_attention_heads,
            attn.d_head,
        )
        .permute(
            0,
            2,
            1,
            3,
        )
    )

    k = (
        k.view(
            b,
            s,
            attn.num_attention_heads,
            attn.d_head,
        )
        .permute(
            0,
            2,
            1,
            3,
        )
    )

    v = (
        v.view(
            b,
            s,
            attn.num_attention_heads,
            attn.d_head,
        )
        .permute(
            0,
            2,
            1,
            3,
        )
    )

    q = justnorm(q)
    k = justnorm(k)

    q = attn.RoPE(q)
    k = attn.RoPE(k)

    sqk = (
        attn.sqk
        * (
            attn.ngpt_sqk_init_value
            / attn.ngpt_sqk_init_scale
        )
    )

    sqk = (
        sqk.view(
            1,
            attn.num_attention_heads,
            1,
            attn.d_head,
        )
        .to(
            q.dtype
        )
    )

    q = sqk * q
    k = sqk * k

    context = (
        F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attention_mask.to(
                q.dtype
            ),
            scale=math.sqrt(
                attn.d_head
            ),
        )
    )

    if (
        attn.config
        .use_exclusive_attention
    ):
        vn = F.normalize(
            v,
            dim=-1,
        )

        context = (
            context
            - (
                context
                * vn
            ).sum(
                dim=-1,
                keepdim=True,
            )
            * vn
        )

    return context


def head_dynamics_analysis(
    model,
    batches,
    dev: DeviceContext,
    layers: Sequence[int],
    max_batches: int,
    sample_tokens: int,
    output_dir: Path,
    router_stats,
):
    """
    Measure head-bank health without oracle/ablation machinery.

    For each selected layer:
      1. recompute all 32 heads before routing
      2. project each head through its own W_O columns
      3. compute residual RMS / energy per head
      4. compute effective rank for all, permanent, elastic heads
      5. compare elastic activation frequency with residual RMS
    """

    H = model.config.num_attention_heads
    P = model.config.num_permanent_heads

    grams = {
        li: np.zeros(
            (H, H),
            dtype=np.float64,
        )
        for li in layers
    }

    sample_count = {
        li: 0
        for li in layers
    }

    with torch.no_grad():
        for cpu_batch in batches[
            :max_batches
        ]:
            batch = move_batch(
                cpu_batch,
                dev,
            )

            with AttentionInputCapture(
                model,
                layers,
            ) as cap:
                with dev.autocast():
                    _ = model(
                        input_ids=batch[
                            "input_ids"
                        ],
                        attention_mask=batch[
                            "attention_mask"
                        ],
                        current_step=6500,
                        easiness_score=batch[
                            "easiness_score"
                        ],
                    )

                dev.mark_step()

            for li in layers:
                hidden, attn_mask = (
                    cap.data[li]
                )

                attn = model.model.blocks[
                    li
                ].attn

                with dev.autocast():
                    context = (
                        unmasked_attention_context(
                            attn,
                            hidden,
                            attn_mask,
                        )
                    )

                    seq = context.size(2)

                    t = min(
                        sample_tokens,
                        seq,
                    )

                    positions = torch.linspace(
                        0,
                        seq - 1,
                        steps=t,
                        device=context.device,
                    ).long()

                    c = context.index_select(
                        2,
                        positions,
                    )

                    # self.output.weight shape:
                    #   [hidden_size, H*d_head]
                    #
                    # Rearrange so each head owns:
                    #   [hidden_size, d_head]
                    W = (
                        attn.output.weight
                        .to(c.dtype)
                        .view(
                            attn.hidden_size,
                            attn.num_attention_heads,
                            attn.d_head,
                        )
                    )

                    # Per-head residual contribution:
                    #   [B,H,T,d] x [D,H,d]
                    #   -> [B,H,T,D]
                    y = torch.einsum(
                        "bhtd,ohd->bhto",
                        c,
                        W,
                    )

                    flat = (
                        y
                        .permute(
                            1,
                            0,
                            2,
                            3,
                        )
                        .contiguous()
                        .view(
                            H,
                            -1,
                        )
                        .float()
                    )

                    gram = (
                        flat
                        @ flat.T
                    )

                dev.mark_step()

                # grams[li] += (
                #     gram
                #     .detach()
                #     .cpu()
                #     .numpy()
                #     .astype(
                #         np.float64
                #     )
                # )
                grams[li] += (gram.detach().to(torch.float32).cpu().numpy().astype(np.float64))


                sample_count[li] += int(
                    flat.shape[1]
                )

                del (
                    context,
                    c,
                    y,
                    flat,
                    gram,
                )

    results = {}

    rows_all = []

    for li in layers:
        gram = grams[li]

        all_erank, all_prank, eig = (
            effective_rank_from_gram(
                gram
            )
        )

        perm_erank, perm_prank, _ = (
            effective_rank_from_gram(
                gram[
                    :P,
                    :P,
                ]
            )
        )

        elastic_erank, elastic_prank, _ = (
            effective_rank_from_gram(
                gram[
                    P:,
                    P:,
                ]
            )
        )

        denom = max(
            1,
            sample_count[li],
        )

        head_energy = np.clip(
            np.diag(gram),
            0.0,
            None,
        )

        head_rms = np.sqrt(
            head_energy
            / float(denom)
        )

        energy_fraction = (
            head_energy
            / (
                head_energy.sum()
                + 1e-12
            )
        )

        elastic_freq = (
            router_stats[
                "masks"
            ][li].mean(
                axis=0
            )
        )

        elastic_rms = (
            head_rms[P:]
        )

        usage_rms_rho = spearman_np(
            elastic_freq,
            elastic_rms,
        )

        usage_energy_rho = spearman_np(
            elastic_freq,
            energy_fraction[
                P:
            ],
        )

        perm_mean_rms = float(
            head_rms[
                :P
            ].mean()
        )

        elastic_mean_rms = float(
            elastic_rms.mean()
        )

        results[li] = {
            "all_entropy_effective_rank":
                all_erank,

            "all_participation_rank":
                all_prank,

            "permanent_entropy_effective_rank":
                perm_erank,

            "permanent_participation_rank":
                perm_prank,

            "elastic_entropy_effective_rank":
                elastic_erank,

            "elastic_participation_rank":
                elastic_prank,

            "permanent_mean_rms":
                perm_mean_rms,

            "elastic_mean_rms":
                elastic_mean_rms,

            "permanent_to_elastic_rms_ratio":
                (
                    perm_mean_rms
                    / (
                        elastic_mean_rms
                        + 1e-12
                    )
                ),

            "activation_rms_spearman":
                usage_rms_rho,

            "activation_energy_spearman":
                usage_energy_rho,

            "head_residual_rms":
                head_rms,

            "head_energy_fraction":
                energy_fraction,
        }

        layer_rows = []

        for h in range(H):
            is_elastic = (
                h >= P
            )

            row = {
                "layer": li,
                "head": h,
                "group":
                    (
                        "elastic"
                        if is_elastic
                        else "permanent"
                    ),
                "residual_rms":
                    float(
                        head_rms[h]
                    ),
                "energy_fraction":
                    float(
                        energy_fraction[h]
                    ),
                "activation_frequency":
                    (
                        float(
                            elastic_freq[
                                h - P
                            ]
                        )
                        if is_elastic
                        else 1.0
                    ),
            }

            layer_rows.append(
                row
            )
            rows_all.append(
                row
            )

        with (
            output_dir
            / f"layer_{li:02d}_head_dynamics.csv"
        ).open(
            "w",
            newline="",
        ) as f:
            writer = csv.DictWriter(
                f,
                fieldnames=layer_rows[0].keys(),
            )

            writer.writeheader()
            writer.writerows(
                layer_rows
            )

        save_bar(
            output_dir
            / f"layer_{li:02d}_residual_rms.png",
            head_rms,
            f"Layer {li}: per-head residual RMS",
            "Residual RMS",
        )

        # Eigen spectrum: useful for seeing whether 7d broadened head energy.
        vals = np.sort(
            np.clip(
                eig,
                0.0,
                None,
            )
        )[::-1]

        frac = (
            vals
            / (
                vals.sum()
                + 1e-12
            )
        )

        fig, ax = plt.subplots(
            figsize=(8, 4)
        )

        ax.plot(
            np.arange(
                1,
                len(frac) + 1,
            ),
            frac,
            marker="o",
        )

        ax.set_title(
            f"Layer {li}: head contribution energy spectrum"
        )

        ax.set_xlabel(
            "Component"
        )

        ax.set_ylabel(
            "Fraction of head-space energy"
        )

        fig.tight_layout()

        fig.savefig(
            output_dir
            / f"layer_{li:02d}_effective_rank_spectrum.png",
            dpi=160,
        )

        plt.close(
            fig
        )

    if rows_all:
        with (
            output_dir
            / "head_dynamics_all_layers.csv"
        ).open(
            "w",
            newline="",
        ) as f:
            writer = csv.DictWriter(
                f,
                fieldnames=rows_all[0].keys(),
            )

            writer.writeheader()
            writer.writerows(
                rows_all
            )

    return results


# -----------------------------------------------------------------------------
# Reporting
# -----------------------------------------------------------------------------
def json_safe(obj):
    if isinstance(obj, dict):
        return {
            str(k): json_safe(v)
            for k, v in obj.items()
        }

    if isinstance(
        obj,
        (list, tuple),
    ):
        return [
            json_safe(v)
            for v in obj
        ]

    if isinstance(
        obj,
        np.ndarray,
    ):
        return obj.tolist()

    if isinstance(
        obj,
        (np.floating, np.integer),
    ):
        return obj.item()

    if (
        isinstance(
            obj,
            float,
        )
        and not np.isfinite(
            obj
        )
    ):
        return None

    return obj


def write_summary(
    output_dir: Path,
    args,
    router_stats,
    ce_modes,
    head_dynamics,
):
    lines = []

    lines.append(
        "# HELM_7d Focused Diagnostic\n"
    )

    lines.append(
        f"Checkpoint: `{args.model_repo}/{args.checkpoint}`\n"
    )

    lines.append(
        f"Examples: {args.num_examples}, "
        f"sequence length: {args.seq_len}, "
        f"batch size: {args.batch_size}\n"
    )

    lines.append(
        "\n## 1. CE comparisons\n"
    )

    for key, value in ce_modes.items():
        lines.append(
            f"- **{key}**: {value:.6f}\n"
        )

    routed = ce_modes[
        "routed"
    ]

    dense = ce_modes[
        "forced_dense"
    ]

    random_ce = ce_modes[
        "random_same_count_mean"
    ]

    if (
        routed
        + 1e-4
        < random_ce
    ):
        lines.append(
            "- Routed still beats random-same-count: "
            "head identity selection remains meaningful.\n"
        )
    else:
        lines.append(
            "- Routed no longer clearly beats random-same-count: "
            "7d may have weakened identity specialization.\n"
        )

    lines.append(
        f"- Forced-dense minus routed CE: "
        f"{dense - routed:+.6f}\n"
    )

    lines.append(
        "\n## 2. Router/cardinality and 7d backward rescue\n"
    )

    lines.append(
        f"- Mean model count loss on diagnostic batches: "
        f"{router_stats['model_count_loss_mean']:.6f}\n"
    )

    for li in range(
        len(
            router_stats[
                "per_layer"
            ]
        )
    ):
        s = router_stats[
            "per_layer"
        ][li]

        lines.append(
            f"- L{li:02d}: "
            f"actual={s['actual_mean']:.2f}, "
            f"target={s['target_mean']:.2f}, "
            f"error={s['count_error_mean']:+.2f}, "
            f"MAE={s['count_mae']:.2f}, "
            f"count_loss={s['layer_count_loss_mean']:.4f}, "
            f"elastic_on={100*s['elastic_on_fraction']:.1f}%, "
            f"sigmoid_mean={s['sigmoid_mean']:.3f}, "
            f"<.45={100*s['sigmoid_below_45']:.1f}%, "
            f".45-.55={100*s['sigmoid_near_half_45_55']:.1f}%, "
            f">.55={100*s['sigmoid_above_55']:.1f}%, "
            f"off-backward={s['off_head_backward_mean']:.3f}, "
            f"off>.45={100*s['off_head_backward_gt_45']:.1f}%, "
            f"STE'={s['ste_derivative_mean']:.3f}, "
            f"Wnorm={s['router_weight_norm_mean']:.2f}\n"
        )

    lines.append(
        "\n### What matters for the 7d failure\n"
    )

    lines.append(
        "- Positive count error together with sigmoid mass shifting above 0.5 "
        "means the router is activating more elastic heads than its easiness target requests.\n"
    )

    lines.append(
        "- A high `off-backward` value means heads that are nominally OFF are "
        "still receiving a large fraction of full CE training. If this approaches ~0.5, "
        "the 7d rescue is not a weak rescue anymore; it is nearly half-strength dense training.\n"
    )

    lines.append(
        "- The key question is whether that extra training actually broadened the elastic head bank. "
        "That is what the effective-rank/RMS section tests.\n"
    )

    lines.append(
        "\n## 3. Head-bank dynamics\n"
    )

    P = 8
    E = 24

    for li, r in head_dynamics.items():
        lines.append(
            f"- L{li:02d}: "
            f"all-rank={r['all_entropy_effective_rank']:.2f}/32, "
            f"perm-rank={r['permanent_entropy_effective_rank']:.2f}/{P}, "
            f"elastic-rank={r['elastic_entropy_effective_rank']:.2f}/{E}, "
            f"perm_RMS={r['permanent_mean_rms']:.6f}, "
            f"elastic_RMS={r['elastic_mean_rms']:.6f}, "
            f"perm/elastic={r['permanent_to_elastic_rms_ratio']:.2f}x, "
            f"rho(activation,RMS)={r['activation_rms_spearman']:.3f}\n"
        )

    lines.append(
        "\n### How to interpret 7d vs 7c\n"
    )

    lines.append(
        "- If elastic effective rank rises and permanent/elastic RMS imbalance falls, "
        "7d successfully reduced starvation even if routing became unstable.\n"
    )

    lines.append(
        "- If elastic rank/RMS barely improve but router counts shift upward, "
        "the rescue path mostly destabilized routing without fixing the head bank.\n"
    )

    lines.append(
        "- If `rho(activation,RMS)` falls substantially from 7c, that supports the "
        "idea that previously rare heads are catching up. If it remains very high, "
        "the rich-get-richer pattern persists.\n"
    )

    lines.append(
        "- If CE becomes noisy while counts overshoot, the most likely interpretation is "
        "that CE increasingly rewards newly-trained elastic heads while the count loss pushes "
        "their logits back down, creating competing objectives around the threshold.\n"
    )

    (
        output_dir
        / "summary.md"
    ).write_text(
        "".join(
            lines
        )
    )


def main():
    parser = argparse.ArgumentParser(
        description="Focused HELM_7d head-training diagnostic"
    )

    parser.add_argument(
        "--model-repo",
        default=MODEL_REPO,
    )

    parser.add_argument(
        "--checkpoint",
        default=CHECKPOINT_FILE,
    )

    parser.add_argument(
        "--data-repo",
        default=DATA_REPO,
    )

    parser.add_argument(
        "--validation-file",
        default=VALIDATION_FILE,
    )

    parser.add_argument(
        "--device",
        choices=[
            "auto",
            "cuda",
            "xla",
            "cpu",
        ],
        default="auto",
    )

    parser.add_argument(
        "--num-examples",
        type=int,
        default=16,
    )

    parser.add_argument(
        "--batch-size",
        type=int,
        default=2,
    )

    parser.add_argument(
        "--seq-len",
        type=int,
        default=1024,
    )

    parser.add_argument(
        "--layers",
        default="0,5,11",
        help="Layers for head-bank dynamics",
    )

    parser.add_argument(
        "--functional-batches",
        type=int,
        default=4,
    )

    parser.add_argument(
        "--functional-sample-tokens",
        type=int,
        default=16,
    )

    parser.add_argument(
        "--random-trials",
        type=int,
        default=3,
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=67,
    )

    parser.add_argument(
        "--cache-dir",
        default="./helm7d_analysis_cache",
    )

    parser.add_argument(
        "--output-dir",
        default="./helm7d_focused_analysis",
    )

    args = parser.parse_args()

    selected_layers = [
        int(x.strip())
        for x in args.layers.split(",")
        if x.strip()
    ]

    output_dir = Path(
        args.output_dir
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    cache_dir = Path(
        args.cache_dir
    )

    seed_everything(
        args.seed
    )

    print(
        "=" * 80
    )

    print(
        "HELM_7d FOCUSED DIAGNOSTIC"
    )

    print(
        "=" * 80
    )

    print(
        "Read-only checkpoint analysis; no parameter updates.\n"
    )

    token = get_hf_token()

    (
        ckpt_path,
        training_state_path,
        validation_path,
    ) = download_assets(
        cache_dir,
        token,
        args.model_repo,
        args.checkpoint,
        args.data_repo,
        args.validation_file,
    )

    breakpoints = load_breakpoints(
        training_state_path
    )

    dev = resolve_device(
        args.device
    )

    print(
        f"Analysis device: "
        f"{dev.device} ({dev.kind})"
    )

    model, config = load_model(
        ckpt_path,
        breakpoints,
        dev,
    )

    if breakpoints is None:
        vals = (
            pq.read_table(
                str(
                    validation_path
                ),
                columns=[
                    "easiness_score"
                ],
            )
            .column(
                "easiness_score"
            )
            .to_numpy(
                zero_copy_only=False
            )
        )

        vals = np.asarray(
            vals,
            dtype=np.float64,
        )

        vals = vals[
            np.isfinite(
                vals
            )
        ]

        breakpoints = np.quantile(
            vals,
            np.linspace(
                0.0,
                1.0,
                101,
            ),
        ).tolist()

        model.config.easiness_cdf_breakpoints = (
            breakpoints
        )

        for block in model.model.blocks:
            block.mlt_vw_rtr.config.easiness_cdf_breakpoints = (
                breakpoints
            )

        print(
            "Fallback: estimated easiness breakpoints "
            "from validation data."
        )

    batches = prepare_batches(
        validation_path,
        config,
        args.num_examples,
        args.batch_size,
        args.seq_len,
        args.seed,
    )

    print(
        "\n[1/3] Router/cardinality + 7d backward-rescue statistics..."
    )

    router_stats = collect_router_statistics(
        model,
        batches,
        dev,
        output_dir,
        selected_layers,
    )

    print(
        f"  Routed CE: "
        f"{router_stats['routed_ce']:.6f}"
    )

    print(
        f"  Mean count loss: "
        f"{router_stats['model_count_loss_mean']:.6f}"
    )

    print(
        "\n[2/3] Controlled CE comparisons..."
    )

    ce_modes = {
        "routed":
            router_stats[
                "routed_ce"
            ],

        "forced_dense":
            evaluate_override_mode(
                model,
                batches,
                dev,
                "dense",
                args.seed,
            ),

        "permanent_only":
            evaluate_override_mode(
                model,
                batches,
                dev,
                "permanent_only",
                args.seed,
            ),
    }

    random_vals = []

    for trial in range(
        args.random_trials
    ):
        value = evaluate_override_mode(
            model,
            batches,
            dev,
            "random_same_count",
            args.seed
            + 1000
            + trial,
        )

        random_vals.append(
            value
        )

        print(
            f"  random same-count {trial}: "
            f"{value:.6f}"
        )

    ce_modes[
        "random_same_count_mean"
    ] = float(
        np.mean(
            random_vals
        )
    )

    ce_modes[
        "random_same_count_std"
    ] = float(
        np.std(
            random_vals
        )
    )

    print(
        "  CE modes:",
        ce_modes,
    )

    print(
        "\n[3/3] Head-bank RMS / effective-rank dynamics..."
    )

    head_dynamics = head_dynamics_analysis(
        model,
        batches,
        dev,
        selected_layers,
        min(
            args.functional_batches,
            len(batches),
        ),
        args.functional_sample_tokens,
        output_dir,
        router_stats,
    )

    for li, r in head_dynamics.items():
        print(
            f"  L{li}: "
            f"elastic rank="
            f"{r['elastic_entropy_effective_rank']:.2f}/24 | "
            f"perm/elastic RMS="
            f"{r['permanent_to_elastic_rms_ratio']:.2f}x | "
            f"rho(use,RMS)="
            f"{r['activation_rms_spearman']:.3f}"
        )

    summary_payload = {
        "args":
            vars(args),

        "checkpoint":
            f"{args.model_repo}/{args.checkpoint}",

        "ce_modes":
            ce_modes,

        "router":
            {
                "routed_ce":
                    router_stats[
                        "routed_ce"
                    ],

                "model_count_loss_mean":
                    router_stats[
                        "model_count_loss_mean"
                    ],

                "per_layer":
                    router_stats[
                        "per_layer"
                    ],
            },

        "head_dynamics":
            head_dynamics,
    }

    with (
        output_dir
        / "summary.json"
    ).open(
        "w"
    ) as f:
        json.dump(
            json_safe(
                summary_payload
            ),
            f,
            indent=2,
        )

    write_summary(
        output_dir,
        args,
        router_stats,
        ce_modes,
        head_dynamics,
    )

    # Archive outside output_dir to avoid self-recursive ZIP growth.
    archive_base = (
        output_dir.parent
        / f"{output_dir.name}_results"
    )

    archive_path = shutil.make_archive(
        str(
            archive_base
        ),
        "zip",
        root_dir=output_dir,
    )

    print(
        "\n"
        + "=" * 80
    )

    print(
        "ANALYSIS COMPLETE"
    )

    print(
        f"Summary: "
        f"{output_dir / 'summary.md'}"
    )

    print(
        f"JSON:    "
        f"{output_dir / 'summary.json'}"
    )

    print(
        f"ZIP:     "
        f"{archive_path}"
    )

    print(
        "Send me summary.md + the ZIP."
    )

    print(
        "=" * 80
    )


if __name__ == "__main__":
    main()

Overwriting analyze_helm7c_heads.py


In [9]:
!python analyze_helm7c_heads.py \
    --device xla \
    --batch-size 2 \
    --num-examples 8 \
    --functional-batches 4

HELM_7d FOCUSED DIAGNOSTIC
Read-only checkpoint analysis; no parameter updates.

Loaded 101 easiness CDF breakpoints from training_state.json
/kaggle/working/analyze_helm7c_heads.py:305: DeprecationWarning: Use torch_xla.device instead
  xm.xla_device(),
E0000 00:00:1786456705.047448    2068 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:238
Analysis device: xla:0 (xla)
Loading checkpoint from /kaggle/working/helm7d_analysis_cache/model_repo/checkpoint-006500.pt
Prepared 8 examples -> 4 batches of 2, seq_len=1024

[1/3] Router/cardinality + 7d backward-rescue statistics...
/kaggle/working/analyze_helm7c_heads.py:267: DeprecationWarning: Use torch_xla.sync instead
  self.xm.mark_step()
  Routed CE: 3.639678
  Mean count loss: 0.029442

[2/3] Controlled CE comparisons...
/kaggle/working/an